# Experiment 5: Controlled Comparison on a Second Dataset (Coveo SIGIR eCom 2021)

Exp5 repeats the focused Exp4 comparison on a different retailer, using the Coveo SIGIR eCom 2021 Data Challenge
browsing data. The task, labels, prefix protocol, models, training settings, class weighting, early stopping and
evaluation are the same as in Exp4. Only the data stage differs:

| | Exp4 (GA4 Merchandise Store) | Exp5 (Coveo) |
|---|---|---|
| Event tokens | 9 GA4 commerce events | pageview, detail, add, remove, purchase |
| Page | URL path | `hashed_url` |
| Item | product in the event's own `items` array | `product_sku_hash` of the product event |
| Item relations | view_item_of, add_to_cart_of, purchase_item_of | detail_of, add_of, remove_of, purchase_of |
| Labels | Buyer = purchase; Intent = cart/checkout; Researcher = ≥3 product interactions | Buyer = purchase; Intent = add; Researcher = ≥3 detail events |
| Session context | 6 GA4 categoricals + is_new_user | start hour, start weekday (no device/traffic fields; is_new_user = 0) |
| Sample | all sessions, T_MAX = 43 | ~300k most recent sessions of a contiguous period; T_MAX = p95 of training lengths |

Stages (each resumable; finished outputs on Google Drive are skipped):
1. **Data**: unzip, sample with DuckDB, build the prefix tables.
2. **Cheap baselines**: Markov-3 and LightGBM.
3. **Neural models** (focused subset): SASRec+Time+SF+PageItem, GRU+Time+SF+PageItem, HGNN_full, HGNN_full_3layer_meanpool, HomoGraphSAGE_3layer_meanpool.
4. **Evaluation**: as in Exp4, with bootstrap tables vs HGNN_full, the GRU and the homogeneous pooled graph.

**Before you start:** register at Coveo and download the dataset zip (research use only, see its Terms and Conditions),
then upload the zip unchanged to `MyDrive/exp5_coveo/raw/` on Google Drive.
**Runtime:** a GPU (L4/A100 recommended, T4 works). High-RAM recommended for stage 1.

**Top-up to 5 seeds (all stages).** `SEEDS` now has 5 seeds. Every stage skips finished runs, so only seeds
789 and 1011 train. Stage 5 already has 5 seeds. Recommended order: setup cells → stage 2 (2 extra LightGBM
seeds, minutes) → stage 5 (no training; copies the new LightGBM seeds and re-evaluates) → stage 3 (fixed
learning rate) → stage 4. Stage 3 copies a run from `lr_selection/final/` instead of retraining when stage 5
chose the default rate for that model (here `HGNN_full`).

## Configuration

In [ ]:
NOTEBOOK_VERSION = "2026-11-01.1"   # must match exp4lib.__version__ printed after the library cells
print("Exp5 notebook version:", NOTEBOOK_VERSION)

# ---- edit these -------------------------------------------------------------
DRIVE_DIR = "/content/drive/MyDrive/exp5_coveo"   # put the Coveo zip into DRIVE_DIR/raw/
N_SESSIONS = 300_000                  # sessions sampled before the T_MAX filter
END_GAP_DAYS = 1.0                    # sample ends this many days before the last session start
SEEDS = [42, 123, 456, 789, 1011]    # 5 seeds in every stage (789/1011 top up the earlier 3-seed runs)

NEURAL_MODELS = [
    "HGNN_full",
    "HGNN_full_3layer_meanpool",
    "HomoGraphSAGE_3layer_meanpool",
    "GRU+Time+SF+PageItem",
    "SASRec+Time+SF+PageItem",
]
LR_GRID = [1e-3, 3e-4, 1e-4]           # optional stage 5: learning rate chosen on validation (seed SEEDS[0])
LR_MODELS = NEURAL_MODELS             # all 5 neural models
LR_SEEDS = SEEDS                       # seeds trained at the chosen learning rate
LGBM_TRIALS = 30
LGBM_TUNE_FRAC = 1.0                  # use 0.5 on a 2-vCPU T4 runtime
LGBM_TUNE_TIMEOUT_SEC = 3 * 3600
N_BOOT = 1000
# -------------------------------------------------------------------------------

## Setup

In [ ]:
!pip -q install torch_geometric optuna lightgbm duckdb
import os, sys, json, time
import torch, psutil
from google.colab import drive
drive.mount("/content/drive")

RAW_DIR = os.path.join(DRIVE_DIR, "raw")
WORK_DIR = "/content/coveo_raw"           # unzipped CSVs on fast local disk (not Drive)
PREP_DIR = os.path.join(DRIVE_DIR, "prepared")
RUN_DIR = os.path.join(DRIVE_DIR, "runs")
RESULTS_DIR = os.path.join(DRIVE_DIR, "results")
for d in (RAW_DIR, PREP_DIR, RUN_DIR, RESULTS_DIR):
    os.makedirs(d, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")
print(f"RAM: {psutil.virtual_memory().total / 1e9:.1f} GB, vCPU: {os.cpu_count()}")
print("Files in raw/:", os.listdir(RAW_DIR))

LOG_PATH = os.path.join(DRIVE_DIR, "log.txt")
def log(msg):
    line = f"{time.strftime('%Y-%m-%d %H:%M:%S')}  {msg}"
    print(line, flush=True)
    with open(LOG_PATH, "a") as f:
        f.write(line + "\n")

### Library
The next cells write the shared Exp4/Exp5 code to `/content/exp4lib/`. Re-run them after every runtime restart.

In [ ]:
os.makedirs('/content/exp4lib', exist_ok=True)

In [ ]:
%%writefile /content/exp4lib/__init__.py
"""Exp4/Exp5 controlled comparison library (written to disk by the notebooks)."""
__version__ = "2026-11-01.1"


In [ ]:
%%writefile /content/exp4lib/data.py
"""Exp4 data stage: BigQuery extraction, Exp3-identical sessions/labels/split,
prefix tables, the item-join leakage check and the fixed per-event item join.

Conventions
-----------
* Events are sorted by (session_id, event_timestamp, event_bundle_sequence_id,
  event_name, row_hash). Exp3 sorted by the first three keys only and relied on
  BigQuery output order for ties, which is not reproducible; event_name is the
  Exp2 tie-breaker and row_hash makes the order fully deterministic.
* Because every session has length <= T_MAX, prefix (s, t) <-> event row
  start[s] + t - 1. All prefix-level arrays are therefore aligned with the
  event table: prefix row i == event row i.
* Labels: Browser=0, Researcher=1, Intent=2, Buyer=3 (Exp3 LABEL_MAP).
"""
import json
import os
from urllib.parse import urlparse

import numpy as np
import pandas as pd

T_MAX = 43
LABEL_MAP = {"Browser": 0, "Researcher": 1, "Intent": 2, "Buyer": 3}
CLASS_ORDER = ["Browser", "Researcher", "Intent", "Buyer"]
SESSION_CAT_COLS = ["device_category", "traffic_source", "traffic_medium",
                    "traffic_campaign", "country", "region"]
ITEM_EVENTS = ["view_item", "add_to_cart", "purchase"]
ITEM_REL = {"view_item": 0, "add_to_cart": 1, "purchase": 2}
REL_NAMES = ["view_item_of", "add_to_cart_of", "purchase_item_of"]
KEY_EVENTS = ["purchase", "add_to_cart", "begin_checkout",
              "add_payment_info", "add_shipping_info"]
PRODUCT_EVENTS_OBSERVABLE = ["view_item", "view_item_list"]

# Expected counts from the Exp3 notebook (Sections 0-2)
EXP3_EXPECTED = {
    "items_base_rows": 3_431_675,
    "select_item_rows": 31_007,
    "events_base_rows": 1_899_352,
    "sessions_before_tmax": 334_037,
    "sessions": 327_283,
    "sessions_by_split": {"train": 229_098, "val": 49_092, "test": 49_093},
    "session_labels": {"Browser": 293_699, "Researcher": 19_048, "Intent": 12_241, "Buyer": 2_295},
    "prefixes": 1_277_287,
    "prefixes_by_split": {"train": 875_203, "val": 190_807, "test": 211_277},
    "test_prefixes_by_class": {"Browser": 131_685, "Buyer": 9_848, "Intent": 29_747, "Researcher": 39_997},
}

_SESSION_ID_SQL = "(SELECT MAX(value.int_value) FROM UNNEST(event_params) WHERE key = 'ga_session_id')"

EVENTS_QUERY = f"""
SELECT
    e.event_timestamp,
    e.event_bundle_sequence_id,
    e.event_name,
    e.user_pseudo_id,
    {_SESSION_ID_SQL} AS ga_session_id,
    (SELECT MAX(value.int_value) FROM UNNEST(event_params) WHERE key = 'ga_session_number') AS ga_session_number,
    (SELECT value.string_value FROM UNNEST(event_params) WHERE key = 'page_location') AS page_location,
    e.device.category       AS device_category,
    e.geo.country           AS country,
    e.geo.region            AS region,
    e.traffic_source.source AS traffic_source,
    e.traffic_source.medium AS traffic_medium,
    e.traffic_source.name   AS traffic_campaign,
    FARM_FINGERPRINT(TO_JSON_STRING(e)) AS row_hash,
    -- items carried on the event row itself: this is the fix for the Exp3 join
    IF(e.event_name IN ('view_item', 'add_to_cart', 'purchase'),
       ARRAY(SELECT AS STRUCT it.item_id, it.price FROM UNNEST(e.items) AS it
             WHERE it.item_id IS NOT NULL),
       []) AS items
FROM `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*` AS e
WHERE e.event_name IN (
    'page_view', 'view_search_results', 'view_item_list', 'view_item',
    'add_to_cart', 'begin_checkout',
    'add_shipping_info', 'add_payment_info', 'purchase'
)
  AND e.user_pseudo_id IS NOT NULL
  AND {_SESSION_ID_SQL} IS NOT NULL
  AND e.event_name IS NOT NULL
"""

SELECT_ITEM_QUERY = f"""
SELECT
    user_pseudo_id,
    {_SESSION_ID_SQL} AS ga_session_id,
    event_timestamp,
    event_name
FROM `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*`
WHERE event_name = 'select_item'
  AND user_pseudo_id IS NOT NULL
  AND {_SESSION_ID_SQL} IS NOT NULL
"""


# --------------------------------------------------------------------------
# Section 0: extraction
# --------------------------------------------------------------------------
def _query_df(client, sql):
    job = client.query(sql)
    try:
        return job.to_dataframe(create_bqstorage_client=True)
    except Exception as e:  # sandbox projects may not have the Storage Read API
        print(f"  (BigQuery Storage API unavailable: {type(e).__name__}; using REST download)")
        return job.result().to_dataframe(create_bqstorage_client=False)


def extract_raw(client, out_dir):
    """Run the two queries and save raw parquet files (events with items flattened out)."""
    ev_path = os.path.join(out_dir, "raw_events.parquet")
    it_path = os.path.join(out_dir, "raw_items.parquet")
    si_path = os.path.join(out_dir, "raw_select_item.parquet")
    if all(os.path.exists(p) for p in (ev_path, it_path, si_path)):
        print("Raw extracts already on Drive, skipping BigQuery.")
        return

    print("Querying events (with per-event items arrays)...")
    ev = _query_df(client, EVENTS_QUERY)
    print(f"  events: {ev.shape}")

    # Flatten items, keeping the event's raw row id as the join key
    ev = ev.reset_index(drop=True)
    ev["raw_row"] = np.arange(len(ev), dtype=np.int64)
    has = ev["items"].map(lambda a: a is not None and len(a) > 0).to_numpy()
    rows, ids, prices = [], [], []
    for r, arr in zip(ev.loc[has, "raw_row"].to_numpy(), ev.loc[has, "items"].to_numpy()):
        for it in arr:
            rows.append(r)
            ids.append(it["item_id"])
            prices.append(it["price"])
    items = pd.DataFrame({"raw_row": np.asarray(rows, dtype=np.int64),
                          "item_id": ids,
                          "price": pd.array(prices, dtype="Float64").astype("float64")})
    ev = ev.drop(columns=["items"])
    print(f"  items (flattened): {items.shape}")

    print("Querying select_item (labels only)...")
    si = _query_df(client, SELECT_ITEM_QUERY)
    print(f"  select_item: {si.shape}")

    ev.to_parquet(ev_path, index=False)
    items.to_parquet(it_path, index=False)
    si.to_parquet(si_path, index=False)


# --------------------------------------------------------------------------
# Section 1-2: sessions, labels, split, vocabularies, prefix tables
# --------------------------------------------------------------------------
def _normalize_page(url):
    if url is None or (isinstance(url, float) and np.isnan(url)):
        return None
    try:
        return urlparse(url).path.rstrip("/") or "/"
    except Exception:
        return None


def _check(name, got, expected, report):
    ok = got == expected
    report.append({"check": name, "got": got, "expected": expected, "ok": bool(ok)})
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}: got={got} expected={expected}")


def build_dataset(raw_dir, out_dir, strict=True):
    """Build all Exp4 tables. Writes to out_dir and returns the repro report."""
    ev = pd.read_parquet(os.path.join(raw_dir, "raw_events.parquet"))
    items = pd.read_parquet(os.path.join(raw_dir, "raw_items.parquet"))
    si = pd.read_parquet(os.path.join(raw_dir, "raw_select_item.parquet"))
    report = []
    print("Reproduction checks against Exp3:")
    _check("events_base_rows", int(len(ev)), EXP3_EXPECTED["events_base_rows"], report)
    _check("items_base_rows", int(len(items)), EXP3_EXPECTED["items_base_rows"], report)
    _check("select_item_rows", int(len(si)), EXP3_EXPECTED["select_item_rows"], report)

    # 1.0 session id + deterministic sort
    ev["session_id"] = ev["user_pseudo_id"] + "-" + ev["ga_session_id"].astype(str)
    ev = ev.sort_values(["session_id", "event_timestamp", "event_bundle_sequence_id",
                         "event_name", "row_hash"], kind="mergesort").reset_index(drop=True)
    ev["event_index"] = ev.groupby("session_id").cumcount() + 1

    # 1.1 session aggregates
    sess = ev.groupby("session_id").agg(
        session_length=("event_index", "max"),
        session_start_ts=("event_timestamp", "min"),
        session_end_ts=("event_timestamp", "max"),
    ).reset_index()

    # 1.2 labels (vectorised version of Exp3 assign_label)
    en = ev["event_name"]
    flags = pd.DataFrame({
        "session_id": ev["session_id"],
        "buy": en.eq("purchase"),
        "intent": en.isin(["add_to_cart", "begin_checkout", "add_payment_info", "add_shipping_info"]),
        "prod": en.isin(["view_item", "view_item_list"]),
    }).groupby("session_id").agg(buy=("buy", "any"), intent=("intent", "any"), prod=("prod", "sum"))
    si["session_id"] = si["user_pseudo_id"] + "-" + si["ga_session_id"].astype(str)
    flags["prod"] = flags["prod"] + si.groupby("session_id").size().reindex(flags.index, fill_value=0)
    y = np.where(flags["buy"], "Buyer",
                 np.where(flags["intent"], "Intent",
                          np.where(flags["prod"] >= 3, "Researcher", "Browser")))
    sess = sess.merge(pd.DataFrame({"session_id": flags.index, "y": y}), on="session_id", how="left")
    _check("sessions_before_tmax", int(len(sess)), EXP3_EXPECTED["sessions_before_tmax"], report)

    # 1.4 T_MAX filter
    sess = sess[sess["session_length"] <= T_MAX].reset_index(drop=True)
    _check("sessions", int(len(sess)), EXP3_EXPECTED["sessions"], report)
    _check("session_labels", {k: int(v) for k, v in sess["y"].value_counts().items()},
           EXP3_EXPECTED["session_labels"], report)

    # 1.5 temporal split (identical to Exp3)
    p70 = int(round(sess["session_end_ts"].quantile(0.70)))
    p85 = int(round(sess["session_end_ts"].quantile(0.85)))
    sess["split"] = np.where(sess["session_end_ts"] <= p70, "train",
                             np.where(sess["session_end_ts"] <= p85, "val", "test"))
    _check("sessions_by_split", {k: int(v) for k, v in sess["split"].value_counts().items()},
           EXP3_EXPECTED["sessions_by_split"], report)

    # 2.0 filter events to kept sessions
    keep = set(sess["session_id"])
    ev = ev[ev["session_id"].isin(keep)].reset_index(drop=True)
    sess = sess.sort_values("session_id").reset_index(drop=True)  # same order as groupby
    sess["session_idx"] = np.arange(len(sess), dtype=np.int32)
    ev = ev.merge(sess[["session_id", "session_idx", "split", "y"]], on="session_id", how="left")
    # merge keeps left order; events are already sorted by session_id
    starts = np.r_[0, np.cumsum(sess["session_length"].to_numpy())[:-1]].astype(np.int64)
    sess["start"] = starts
    assert len(ev) == int(sess["session_length"].sum())
    assert (ev["session_idx"].to_numpy()[starts] == sess["session_idx"].to_numpy()).all()

    n_pref = len(ev)
    _check("prefixes", int(n_pref), EXP3_EXPECTED["prefixes"], report)
    _check("prefixes_by_split", {k: int(v) for k, v in ev["split"].value_counts().items()},
           EXP3_EXPECTED["prefixes_by_split"], report)
    tst = ev[ev["split"] == "test"]["y"].value_counts()
    _check("test_prefixes_by_class", {k: int(v) for k, v in tst.items()},
           EXP3_EXPECTED["test_prefixes_by_class"], report)

    train_mask = ev["split"].eq("train").to_numpy()

    # 2.1 page_key
    uniq_loc = ev["page_location"].dropna().unique()
    loc_map = {u: _normalize_page(u) for u in uniq_loc}
    ev["page_key"] = ev["page_location"].map(loc_map)

    # 2.2 delta_t_prev (microseconds, z-scored with train stats) + Exp2-style seconds
    dt_us = ev.groupby("session_idx")["event_timestamp"].diff().fillna(0).to_numpy(np.float64)
    dt_mean = float(dt_us[train_mask].mean())
    dt_std = float(pd.Series(dt_us[train_mask]).std())
    ev["dt_norm"] = ((dt_us - dt_mean) / dt_std).astype(np.float32)
    dt_sec = dt_us / 1e6
    pos_tr = dt_sec[train_mask & (ev["event_index"].to_numpy() > 1)]
    max_dt = float(np.percentile(pos_tr[pos_tr > 0], 99))  # Exp2 MAX_DT
    ev["dt_sec"] = dt_sec.astype(np.float32)
    ev["dt01"] = (np.minimum(dt_sec, max_dt) / max_dt).astype(np.float32)

    # 2.4 vocabularies (train only)
    tr_ev = ev[train_mask]
    event_vocab = {n: i + 2 for i, n in enumerate(sorted(tr_ev["event_name"].unique()))}
    event_vocab.update({"<PAD>": 0, "<UNK>": 1})
    page_vocab = {k: i + 2 for i, k in enumerate(sorted(tr_ev["page_key"].dropna().unique()))}
    page_vocab.update({"<PAD>": 0, "<OOV>": 1})
    ev["event_name_idx"] = ev["event_name"].map(event_vocab).fillna(1).astype(np.int16)
    ev["page_key_idx"] = ev["page_key"].map(page_vocab).fillna(1).astype(np.int32)

    # items: attach to the filtered/sorted event rows via raw_row (the fixed join)
    raw2row = pd.Series(np.arange(n_pref, dtype=np.int64), index=ev["raw_row"].to_numpy())
    items = items[items["raw_row"].isin(raw2row.index)].copy()
    items["event_row"] = raw2row.reindex(items["raw_row"].to_numpy()).to_numpy()
    items = items.sort_values(["event_row", "raw_row"], kind="mergesort").reset_index(drop=True)
    items_split = ev["split"].to_numpy()[items["event_row"].to_numpy()]
    tr_items = items[items_split == "train"]
    item_vocab = {k: i + 2 for i, k in enumerate(sorted(tr_items["item_id"].unique()))}
    item_vocab.update({"<PAD>": 0, "<OOV>": 1})
    items["item_idx"] = items["item_id"].map(item_vocab).fillna(1).astype(np.int32)

    # 2.3 price normalisation and train-only item node price (Exp3 3.1)
    price_mean = float(tr_items["price"].mean())
    price_std = float(tr_items["price"].std())
    items["price_norm"] = (items["price"].fillna(0) - price_mean) / price_std
    item_price = (items[items_split == "train"].groupby("item_idx")["price_norm"].mean())
    price_lookup = np.zeros(len(item_vocab), dtype=np.float32)  # OOV/PAD -> 0.0
    price_lookup[item_price.index.to_numpy()] = item_price.to_numpy(np.float32)
    for k in ("<PAD>", "<OOV>"):
        price_lookup[item_vocab[k]] = 0.0

    ev_name_arr = ev["event_name"].to_numpy()
    items["rel"] = pd.Series(ev_name_arr[items["event_row"].to_numpy()]).map(ITEM_REL).to_numpy(np.int8)

    # 5.0 session-constant features (first non-null per session, as Exp3 groupby.first())
    sf = ev.groupby("session_idx")[SESSION_CAT_COLS + ["ga_session_number"]].first()
    gsn = pd.to_numeric(sf["ga_session_number"].reindex(sess["session_idx"]), errors="coerce").fillna(-1).to_numpy()
    sess["is_new_user"] = (gsn == 1).astype(np.int8)
    cat_vocabs = {}
    train_sess = sess["split"].eq("train").to_numpy()
    for col in SESSION_CAT_COLS:
        vals = sf[col].reindex(sess["session_idx"]).fillna("<NA>").astype(str).to_numpy()
        vocab = {v: i + 1 for i, v in enumerate(sorted(set(vals[train_sess])))}  # 0 = OOV
        cat_vocabs[col] = vocab
        sess[col + "_idx"] = pd.Series(vals).map(vocab).fillna(0).astype(np.int32).to_numpy()

    # Exp2-style session context strings for LightGBM
    for col in SESSION_CAT_COLS:
        sess["ctx_" + col] = sf[col].reindex(sess["session_idx"]).to_numpy()

    # ------------------------------------------------------------------
    # Item edges: fixed (own event) and Exp3-style (timestamp/bundle join)
    # ------------------------------------------------------------------
    pos = ev["event_index"].to_numpy(np.int16)
    sess_of_row = ev["session_idx"].to_numpy(np.int32)
    key_cols = ["session_idx", "event_timestamp", "event_bundle_sequence_id"]
    evk = ev[key_cols].copy()
    evk["event_row"] = np.arange(n_pref, dtype=np.int64)
    evk["is_item_event"] = ev["event_name"].isin(ITEM_EVENTS).to_numpy()

    it = items[["event_row", "item_idx", "item_id", "rel"]].copy()
    it["item_row"] = np.arange(len(it), dtype=np.int64)
    it = it.join(evk.set_index("event_row")[key_cols], on="event_row")

    # Exp3 graph: every prefix item-event links to every item row with the same key,
    # routed by the PREFIX event's name
    g_ev = evk[evk["is_item_event"]].rename(columns={"event_row": "src_row"})
    exp3_edges = it.merge(g_ev[key_cols + ["src_row"]], on=key_cols, how="inner")
    exp3_edges["src_rel"] = pd.Series(ev_name_arr[exp3_edges["src_row"].to_numpy()]).map(ITEM_REL).to_numpy(np.int8)
    exp3_edges = exp3_edges.sort_values(["src_row", "item_row"], kind="mergesort").reset_index(drop=True)

    # Exp3 aggregates: any prefix event (any name) with the same key pulls item rows in
    any_ev = evk.rename(columns={"event_row": "src_row"})
    agg_links = it.merge(any_ev[key_cols + ["src_row"]], on=key_cols, how="inner")

    def csr(src_rows, n):
        counts = np.bincount(src_rows, minlength=n)
        return np.r_[0, np.cumsum(counts)].astype(np.int64)

    fixed_ptr = csr(items["event_row"].to_numpy(), n_pref)
    exp3_ptr = csr(exp3_edges["src_row"].to_numpy(), n_pref)

    # --- prefix-level unique_items, fixed vs Exp3 ---
    def unique_items_curve(item_ids, eff_row):
        """unique_items at every prefix row given the row at which each item row becomes visible."""
        df = pd.DataFrame({"s": sess_of_row[eff_row], "item": item_ids, "row": eff_row})
        first = df.groupby(["s", "item"])["row"].min().to_numpy()
        inc = np.bincount(first, minlength=n_pref)
        cum = np.cumsum(inc)
        start_of_row = sess["start"].to_numpy()[sess_of_row]
        before = np.r_[0, cum][start_of_row]
        return (cum - before).astype(np.int32)

    uniq_fixed = unique_items_curve(items["item_id"].to_numpy(), items["event_row"].to_numpy())
    first_vis = agg_links.groupby("item_row")["src_row"].min()
    uniq_exp3 = unique_items_curve(it["item_id"].to_numpy()[first_vis.index.to_numpy()],
                                   first_vis.to_numpy())

    # --- leakage accounting ---
    def covered(lo_rows, hi_rows):
        """prefix rows r with lo <= r < hi for any interval (all within one session)."""
        m = lo_rows < hi_rows
        diff = np.zeros(n_pref + 1, dtype=np.int64)
        np.add.at(diff, lo_rows[m], 1)
        np.add.at(diff, hi_rows[m], -1)
        return np.cumsum(diff)[:-1] > 0

    leak_agg = covered(agg_links["src_row"].to_numpy(), agg_links["event_row"].to_numpy())
    leak_graph = covered(exp3_edges["src_row"].to_numpy(), exp3_edges["event_row"].to_numpy())
    signal = exp3_edges[ev_name_arr[exp3_edges["event_row"].to_numpy()] != "view_item"]
    leak_graph_signal = covered(signal["src_row"].to_numpy(), signal["event_row"].to_numpy())
    misrouted_edges = int((exp3_edges["src_rel"].to_numpy() != exp3_edges["rel"].to_numpy()).sum())
    dup_edges = int(len(exp3_edges) - (exp3_edges["src_row"] == exp3_edges["event_row"]).sum())

    # shared-timestamp statistic quoted in the brief
    grp = ev.groupby(["session_idx", "event_timestamp"])["event_index"].transform("size")
    share_ts = float((grp > 1).mean())

    split_arr = ev["split"].to_numpy()
    y_arr = ev["y"].map(LABEL_MAP).to_numpy(np.int8)
    leak = {
        "events_sharing_timestamp_within_session": round(share_ts, 4),
        "item_rows": int(len(items)),
        "exp3_graph_edges": int(len(exp3_edges)),
        "fixed_graph_edges": int(len(items)),
        "exp3_edges_not_from_own_event": dup_edges,
        "exp3_edges_with_wrong_relation_type": misrouted_edges,
        "prefixes_total": int(n_pref),
    }
    for name, mask in [("unique_items_sees_future_items", leak_agg),
                       ("unique_items_value_differs", uniq_exp3 != uniq_fixed),
                       ("graph_has_future_item_edges", leak_graph),
                       ("graph_has_future_add_to_cart_or_purchase_items", leak_graph_signal)]:
        leak[name] = {
            "all": int(mask.sum()),
            "by_split": {s: int(mask[split_arr == s].sum()) for s in ("train", "val", "test")},
            "test_by_class": {c: int(mask[(split_arr == "test") & (y_arr == LABEL_MAP[c])].sum())
                              for c in CLASS_ORDER},
        }
    print("\nLeakage check (Exp3 timestamp/bundle join):")
    print(json.dumps(leak, indent=2))

    # ------------------------------------------------------------------
    # Prefix-level arrays (prefix row == event row)
    # ------------------------------------------------------------------
    en_arr = ev["event_name"]
    cart_flag = en_arr.eq("add_to_cart").astype(np.int8).groupby(ev["session_idx"]).cummax().to_numpy(np.int8)
    key_seen = en_arr.isin(KEY_EVENTS).astype(np.int8).groupby(ev["session_idx"]).cummax().to_numpy() > 0
    prod_cnt = en_arr.isin(PRODUCT_EVENTS_OBSERVABLE).astype(np.int32).groupby(ev["session_idx"]).cumsum().to_numpy()
    undetermined = (~key_seen) & (prod_cnt < 3)

    prefix = pd.DataFrame({
        "session_idx": sess_of_row,
        "session_id": ev["session_id"].to_numpy(),
        "prefix_len": pos,
        "y": y_arr,
        "split": split_arr,
        "unique_items": uniq_fixed,
        "unique_items_exp3": uniq_exp3,
        "cart_flag": cart_flag,
        "undetermined": undetermined,
        "leak_agg": leak_agg,
        "leak_graph": leak_graph,
    })

    events_out = pd.DataFrame({
        "session_idx": sess_of_row,
        "event_index": pos,
        "event_name": ev["event_name"].to_numpy(),
        "event_name_idx": ev["event_name_idx"].to_numpy(),
        "page_key_idx": ev["page_key_idx"].to_numpy(),
        "dt_norm": ev["dt_norm"].to_numpy(),
        "dt_sec": ev["dt_sec"].to_numpy(),
        "dt01": ev["dt01"].to_numpy(),
        "event_timestamp": ev["event_timestamp"].to_numpy(),
    })
    sess_out = sess.drop(columns=[]).copy()

    os.makedirs(out_dir, exist_ok=True)
    events_out.to_parquet(os.path.join(out_dir, "events.parquet"), index=False)
    prefix.to_parquet(os.path.join(out_dir, "prefixes.parquet"), index=False)
    sess_out.to_parquet(os.path.join(out_dir, "sessions.parquet"), index=False)
    items[["event_row", "item_idx", "item_id", "rel"]].to_parquet(os.path.join(out_dir, "items.parquet"), index=False)
    np.savez_compressed(
        os.path.join(out_dir, "edges.npz"),
        fixed_ptr=fixed_ptr,
        fixed_item=items["item_idx"].to_numpy(np.int32),
        fixed_rel=items["rel"].to_numpy(np.int8),
        exp3_ptr=exp3_ptr,
        exp3_item=exp3_edges["item_idx"].to_numpy(np.int32),
        exp3_rel=exp3_edges["src_rel"].to_numpy(np.int8),
        price_lookup=price_lookup,
    )
    meta = {
        "T_MAX": T_MAX, "LABEL_MAP": LABEL_MAP, "CLASS_ORDER": CLASS_ORDER,
        "split_boundaries": {"p70": p70, "p85": p85},
        "dt_stats": {"dt_mean_us": dt_mean, "dt_std_us": dt_std, "MAX_DT_sec": max_dt},
        "price_stats": {"price_mean": price_mean, "price_std": price_std},
        "vocab_sizes": {"event": len(event_vocab), "page": len(page_vocab), "item": len(item_vocab)},
        "event_vocab": event_vocab,
        "cat_vocab_sizes": {c: len(v) for c, v in cat_vocabs.items()},
        "repro_report": report,
        "repro_all_pass": all(r["ok"] for r in report),
        "leakage": leak,
    }
    with open(os.path.join(out_dir, "meta.json"), "w") as f:
        json.dump(meta, f, indent=2, default=int)
    with open(os.path.join(out_dir, "cat_vocabs.json"), "w") as f:
        json.dump(cat_vocabs, f)

    if strict and not meta["repro_all_pass"]:
        bad = [r["check"] for r in report if not r["ok"]]
        raise AssertionError(f"Exp3 counts not reproduced: {bad}. Set STRICT_REPRO=False to continue anyway.")
    return meta


# --------------------------------------------------------------------------
# Loading for the model stages
# --------------------------------------------------------------------------
class Exp4Data:
    """Light numpy view of the prepared tables, shared by every model."""

    def __init__(self, data_dir):
        with open(os.path.join(data_dir, "meta.json")) as f:
            self.meta = json.load(f)
        ev = pd.read_parquet(os.path.join(data_dir, "events.parquet"))
        pf = pd.read_parquet(os.path.join(data_dir, "prefixes.parquet"))
        ss = pd.read_parquet(os.path.join(data_dir, "sessions.parquet"))
        e = np.load(os.path.join(data_dir, "edges.npz"))

        self.n = len(ev)
        self.ev_name = ev["event_name_idx"].to_numpy(np.int64)
        self.ev_page = ev["page_key_idx"].to_numpy(np.int64)
        self.ev_dtn = ev["dt_norm"].to_numpy(np.float32)
        self.ev_dt01 = ev["dt01"].to_numpy(np.float32)
        self.ev_pos = ev["event_index"].to_numpy(np.int64)
        self.ev_sess = ev["session_idx"].to_numpy(np.int64)
        self.sess_start = ss["start"].to_numpy(np.int64)
        # dataset-specific settings (defaults = Exp3/Exp4 GA4 dataset)
        self.session_cat_cols = self.meta.get("session_cat_cols", SESSION_CAT_COLS)
        self.key_events = self.meta.get("key_events", KEY_EVENTS)
        self.item_relations = self.meta.get("item_relations", REL_NAMES)
        self.t_max = int(self.meta.get("T_MAX", T_MAX))
        self.cat_cols = [c + "_idx" for c in self.session_cat_cols]
        self.sess_cat = ss[self.cat_cols].to_numpy(np.int64)
        self.sess_new = ss["is_new_user"].to_numpy(np.float32)

        self.y = pf["y"].to_numpy(np.int64)
        self.split = pf["split"].to_numpy()
        self.session_id = pf["session_id"].to_numpy()
        self.prefix_len = pf["prefix_len"].to_numpy(np.int64)
        self.undetermined = pf["undetermined"].to_numpy(bool)
        self.uniq = {"fixed": pf["unique_items"].to_numpy(np.float32),
                     "exp3": pf["unique_items_exp3"].to_numpy(np.float32)}
        self.cart = pf["cart_flag"].to_numpy(np.float32)
        self.edges = {
            "fixed": (e["fixed_ptr"], e["fixed_item"].astype(np.int64), e["fixed_rel"].astype(np.int64)),
            "exp3": (e["exp3_ptr"], e["exp3_item"].astype(np.int64), e["exp3_rel"].astype(np.int64)),
        }
        self.price_lookup = e["price_lookup"].astype(np.float32)
        self.vocab = self.meta["vocab_sizes"]
        self.cat_vocab_sizes = [self.meta["cat_vocab_sizes"][c] for c in self.session_cat_cols]
        self.rows = {s: np.flatnonzero(self.split == s) for s in ("train", "val", "test")}

    def class_weights(self):
        """Prefix-row balanced weights: total / (n_classes * count), Exp3 7.0."""
        ytr = self.y[self.rows["train"]]
        counts = np.bincount(ytr, minlength=4)
        return (len(ytr) / (4 * counts)).astype(np.float32)

    def session_numeric(self, rows, join="fixed"):
        """[prefix_len, unique_items, cart_flag, is_new_user], raw values as in Exp3."""
        return np.stack([self.prefix_len[rows].astype(np.float32),
                         self.uniq[join][rows],
                         self.cart[rows],
                         self.sess_new[self.ev_sess[rows]]], axis=1)


In [ ]:
%%writefile /content/exp4lib/coveo.py
"""Exp5 data stage: the Coveo SIGIR eCom 2021 Data Challenge dataset, prepared into
exactly the same tables as data.build_dataset() so that every Exp4 model, the
training loop and the evaluation can be reused unchanged.

Mapping to the Exp4 protocol
----------------------------
* Event token: product_action (detail / add / remove / purchase), or "pageview"
  for page views without a product.
* Page = hashed_url; Item = product_sku_hash (one item per product event, so the
  item join is per-event by construction: no timestamp join, no leakage).
* Labels (full session, priority order): Buyer = purchase; Intent = add (no
  purchase); Researcher = >= 3 detail events; Browser = the rest.
* Session context (prefix-safe, known at the first event): start hour and start
  weekday. Coveo has no device/traffic fields; timestamps are shifted by whole
  weeks, which keeps intra-week patterns.
* Sample: the N most recent sessions that start at least `end_gap_days` before
  the last session start (a contiguous period). T_MAX = p95 of training session
  lengths, then the Exp3 p70/p85 temporal split by session end.
"""
import glob
import json
import os
import zipfile

import numpy as np
import pandas as pd

from .data import CLASS_ORDER, LABEL_MAP

EVENT_TOKENS = ["add", "detail", "pageview", "purchase", "remove"]
ITEM_RELATIONS = ["detail_of", "add_of", "remove_of", "purchase_of"]
REL_OF_TOKEN = {"detail": 0, "add": 1, "remove": 2, "purchase": 3}
KEY_EVENTS = ["purchase", "add", "remove"]
SESSION_CAT_COLS = ["start_hour", "start_weekday"]
NEEDED_FILES = ["browsing_train.csv", "sku_to_content.csv"]


def locate_raw(raw_dir, work_dir):
    """Return paths to browsing_train.csv and sku_to_content.csv, extracting them from a
    zip in raw_dir (to fast local disk) if they are not already available."""
    os.makedirs(work_dir, exist_ok=True)
    found = {}
    for name in NEEDED_FILES:
        for d in (work_dir, raw_dir):
            hits = glob.glob(os.path.join(d, "**", name), recursive=True)
            if hits:
                found[name] = hits[0]
                break
    missing = [n for n in NEEDED_FILES if n not in found]
    if missing:
        zips = sorted(glob.glob(os.path.join(raw_dir, "*.zip")))
        if not zips:
            raise FileNotFoundError(f"No {missing} and no .zip in {raw_dir}")
        with zipfile.ZipFile(zips[0]) as z:
            for member in z.namelist():
                base = os.path.basename(member)
                if base in missing:
                    print(f"Extracting {member} from {os.path.basename(zips[0])} ...")
                    target = os.path.join(work_dir, base)
                    with z.open(member) as src, open(target, "wb") as dst:
                        while True:
                            buf = src.read(1 << 24)
                            if not buf:
                                break
                            dst.write(buf)
                    found[base] = target
    missing = [n for n in NEEDED_FILES if n not in found]
    if missing:
        raise FileNotFoundError(f"Could not find {missing} in {raw_dir} or its zip")
    return found["browsing_train.csv"], found["sku_to_content.csv"]


def _browsing_view(con, path):
    con.execute(f"""
        CREATE OR REPLACE VIEW browsing AS
        SELECT * FROM read_csv('{path}', header=true, columns={{
            'session_id_hash': 'VARCHAR', 'event_type': 'VARCHAR', 'product_action': 'VARCHAR',
            'product_sku_hash': 'VARCHAR', 'server_timestamp_epoch_ms': 'BIGINT',
            'hashed_url': 'VARCHAR'}})
    """)


def extract_sample(browsing_csv, sku_csv, n_sessions, end_gap_days=1.0):
    """Select a contiguous sample of sessions with DuckDB (streams the 36M-row file)."""
    import duckdb
    con = duckdb.connect()
    _browsing_view(con, browsing_csv.replace("\\", "/"))
    con.execute("""
        CREATE TEMP TABLE sess AS
        SELECT session_id_hash, MIN(server_timestamp_epoch_ms) AS start_ms, COUNT(*) AS n
        FROM browsing GROUP BY 1
    """)
    total_sessions, max_start = con.execute("SELECT COUNT(*), MAX(start_ms) FROM sess").fetchone()
    cutoff = int(max_start - end_gap_days * 86_400_000)
    con.execute(f"""
        CREATE TEMP TABLE chosen AS
        SELECT session_id_hash FROM sess WHERE start_ms <= {cutoff}
        ORDER BY start_ms DESC, session_id_hash LIMIT {int(n_sessions)}
    """)
    ev = con.execute("""
        SELECT b.session_id_hash, b.event_type, b.product_action, b.product_sku_hash,
               b.server_timestamp_epoch_ms AS ts_ms, b.hashed_url,
               hash(b.session_id_hash, b.event_type, b.product_action, b.product_sku_hash,
                    b.server_timestamp_epoch_ms, b.hashed_url) AS row_hash
        FROM browsing b JOIN chosen c USING (session_id_hash)
    """).df()
    sku = con.execute(f"""
        SELECT product_sku_hash, category_hash, price_bucket
        FROM read_csv('{sku_csv.replace(chr(92), "/")}', header=true, columns={{
            'product_sku_hash': 'VARCHAR', 'description_vector': 'VARCHAR',
            'category_hash': 'VARCHAR', 'image_vector': 'VARCHAR', 'price_bucket': 'DOUBLE'}})
        WHERE product_sku_hash IN (SELECT DISTINCT product_sku_hash FROM browsing
                                   JOIN chosen USING (session_id_hash))
    """).df()
    info = {"total_sessions_in_file": int(total_sessions), "sampled_sessions": int(ev["session_id_hash"].nunique()),
            "sample_cutoff_start_ms": cutoff, "end_gap_days": end_gap_days}
    return ev, sku, info


def build_dataset(raw_dir, work_dir, out_dir, n_sessions=300_000, end_gap_days=1.0):
    browsing_csv, sku_csv = locate_raw(raw_dir, work_dir)
    ev, sku, info = extract_sample(browsing_csv, sku_csv, n_sessions, end_gap_days)
    print(f"Sampled {info['sampled_sessions']:,} of {info['total_sessions_in_file']:,} sessions, "
          f"{len(ev):,} events")

    act = ev["product_action"].fillna("").str.strip()
    ev["event_name"] = np.where(act == "", "pageview", act)
    ev["session_id"] = ev["session_id_hash"]
    ev = ev.sort_values(["session_id", "ts_ms", "event_name", "row_hash"], kind="mergesort").reset_index(drop=True)
    ev["event_index"] = ev.groupby("session_id").cumcount() + 1

    sess = ev.groupby("session_id").agg(session_length=("event_index", "max"),
                                        session_start_ts=("ts_ms", "min"),
                                        session_end_ts=("ts_ms", "max")).reset_index()
    flags = pd.DataFrame({"session_id": ev["session_id"], "buy": ev["event_name"].eq("purchase"),
                          "add": ev["event_name"].eq("add"),
                          "det": ev["event_name"].eq("detail").astype(np.int32)}
                         ).groupby("session_id").agg(buy=("buy", "any"), add=("add", "any"), det=("det", "sum"))
    y = np.where(flags["buy"], "Buyer", np.where(flags["add"], "Intent",
                 np.where(flags["det"] >= 3, "Researcher", "Browser")))
    sess = sess.merge(pd.DataFrame({"session_id": flags.index, "y": y}), on="session_id")

    def split_of(frame):
        p70 = int(round(frame["session_end_ts"].quantile(0.70)))
        p85 = int(round(frame["session_end_ts"].quantile(0.85)))
        return np.where(frame["session_end_ts"] <= p70, "train",
                        np.where(frame["session_end_ts"] <= p85, "val", "test")), p70, p85

    prov, _, _ = split_of(sess)
    t_max = int(np.percentile(sess.loc[prov == "train", "session_length"], 95))
    n_before = len(sess)
    sess = sess[sess["session_length"] <= t_max].reset_index(drop=True)
    sess["split"], p70, p85 = split_of(sess)
    print(f"T_MAX = {t_max} (p95 of train lengths); kept {len(sess):,} of {n_before:,} sessions")

    keep = set(sess["session_id"])
    ev = ev[ev["session_id"].isin(keep)].reset_index(drop=True)
    sess = sess.sort_values("session_id").reset_index(drop=True)
    sess["session_idx"] = np.arange(len(sess), dtype=np.int32)
    ev = ev.merge(sess[["session_id", "session_idx", "split", "y"]], on="session_id", how="left")
    sess["start"] = np.r_[0, np.cumsum(sess["session_length"].to_numpy())[:-1]].astype(np.int64)
    n = len(ev)
    assert n == int(sess["session_length"].sum())
    train = ev["split"].eq("train").to_numpy()

    # time features (seconds), Exp3-style z-score and Exp2-style clipped [0, 1]
    dt = ev.groupby("session_idx")["ts_ms"].diff().fillna(0).to_numpy(np.float64) / 1000.0
    dt_mean, dt_std = float(dt[train].mean()), float(pd.Series(dt[train]).std())
    pos = dt[train & (ev["event_index"].to_numpy() > 1)]
    max_dt = float(np.percentile(pos[pos > 0], 99))

    tr = ev[train]
    event_vocab = {k: i + 2 for i, k in enumerate(sorted(tr["event_name"].unique()))}
    event_vocab.update({"<PAD>": 0, "<UNK>": 1})
    page_vocab = {k: i + 2 for i, k in enumerate(sorted(tr["hashed_url"].dropna().unique()))}
    page_vocab.update({"<PAD>": 0, "<OOV>": 1})

    # items: one per product event that carries a SKU
    is_item = ev["event_name"].isin(list(REL_OF_TOKEN)) & ev["product_sku_hash"].notna()
    items = pd.DataFrame({"event_row": np.flatnonzero(is_item.to_numpy()).astype(np.int64),
                          "item_id": ev.loc[is_item, "product_sku_hash"].to_numpy(),
                          "rel": ev.loc[is_item, "event_name"].map(REL_OF_TOKEN).to_numpy(np.int8)})
    item_train = train[items["event_row"].to_numpy()]
    item_vocab = {k: i + 2 for i, k in enumerate(sorted(items.loc[item_train, "item_id"].unique()))}
    item_vocab.update({"<PAD>": 0, "<OOV>": 1})
    items["item_idx"] = items["item_id"].map(item_vocab).fillna(1).astype(np.int32)
    price = items["item_id"].map(sku.set_index("product_sku_hash")["price_bucket"])
    p_mean, p_std = float(price[item_train].mean()), float(price[item_train].std())
    items["price_norm"] = (price.fillna(0) - p_mean) / p_std
    ip = items[item_train].groupby("item_idx")["price_norm"].mean()
    price_lookup = np.zeros(len(item_vocab), dtype=np.float32)
    price_lookup[ip.index.to_numpy()] = ip.to_numpy(np.float32)
    price_lookup[[0, 1]] = 0.0

    # session context: start hour / weekday (UTC of shifted timestamps)
    start_dt = pd.to_datetime(sess["session_start_ts"], unit="ms", utc=True)
    sess["ctx_start_hour"] = start_dt.dt.hour.astype(str).to_numpy()
    sess["ctx_start_weekday"] = start_dt.dt.weekday.astype(str).to_numpy()
    cat_vocabs = {}
    tr_sess = sess["split"].eq("train").to_numpy()
    for col in SESSION_CAT_COLS:
        vals = sess["ctx_" + col]
        vocab = {v: i + 1 for i, v in enumerate(sorted(set(vals[tr_sess])))}
        cat_vocabs[col] = vocab
        sess[col + "_idx"] = vals.map(vocab).fillna(0).astype(np.int32).to_numpy()
    sess["is_new_user"] = np.int8(0)  # not available in Coveo; constant

    # prefix-level arrays (prefix row == event row)
    sess_of_row = ev["session_idx"].to_numpy(np.int64)
    df = pd.DataFrame({"s": sess_of_row[items["event_row"].to_numpy()], "item": items["item_id"].to_numpy(),
                       "row": items["event_row"].to_numpy()})
    first = df.groupby(["s", "item"])["row"].min().to_numpy()
    cum = np.cumsum(np.bincount(first, minlength=n))
    uniq = (cum - np.r_[0, cum][sess["start"].to_numpy()[sess_of_row]]).astype(np.int32)
    grp = ev["session_idx"]
    en = ev["event_name"]
    cart = en.eq("add").astype(np.int8).groupby(grp).cummax().to_numpy(np.int8)
    key_seen = en.isin(["add", "purchase"]).astype(np.int8).groupby(grp).cummax().to_numpy() > 0
    det_cnt = en.eq("detail").astype(np.int32).groupby(grp).cumsum().to_numpy()
    undetermined = (~key_seen) & (det_cnt < 3)
    split_arr = ev["split"].to_numpy()
    y_arr = ev["y"].map(LABEL_MAP).to_numpy(np.int8)
    false = np.zeros(n, dtype=bool)

    prefix = pd.DataFrame({"session_idx": sess_of_row, "session_id": ev["session_id"].to_numpy(),
                           "prefix_len": ev["event_index"].to_numpy(np.int16), "y": y_arr, "split": split_arr,
                           "unique_items": uniq, "unique_items_exp3": uniq, "cart_flag": cart,
                           "undetermined": undetermined, "leak_agg": false, "leak_graph": false})
    events_out = pd.DataFrame({
        "session_idx": sess_of_row, "event_index": ev["event_index"].to_numpy(np.int16),
        "event_name": en.to_numpy(),
        "event_name_idx": en.map(event_vocab).fillna(1).astype(np.int16).to_numpy(),
        "page_key_idx": ev["hashed_url"].map(page_vocab).fillna(1).astype(np.int32).to_numpy(),
        "dt_norm": ((dt - dt_mean) / dt_std).astype(np.float32), "dt_sec": dt.astype(np.float32),
        "dt01": (np.minimum(dt, max_dt) / max_dt).astype(np.float32),
        "event_timestamp": ev["ts_ms"].to_numpy(np.int64) * 1000,  # microseconds, as in Exp4
    })
    counts = np.bincount(items["event_row"].to_numpy(), minlength=n)
    ptr = np.r_[0, np.cumsum(counts)].astype(np.int64)

    os.makedirs(out_dir, exist_ok=True)
    events_out.to_parquet(os.path.join(out_dir, "events.parquet"), index=False)
    prefix.to_parquet(os.path.join(out_dir, "prefixes.parquet"), index=False)
    sess.to_parquet(os.path.join(out_dir, "sessions.parquet"), index=False)
    items[["event_row", "item_idx", "item_id", "rel"]].to_parquet(os.path.join(out_dir, "items.parquet"), index=False)
    item_arr, rel_arr = items["item_idx"].to_numpy(np.int32), items["rel"].to_numpy(np.int8)
    np.savez_compressed(os.path.join(out_dir, "edges.npz"), fixed_ptr=ptr, fixed_item=item_arr, fixed_rel=rel_arr,
                        exp3_ptr=ptr, exp3_item=item_arr, exp3_rel=rel_arr, price_lookup=price_lookup)

    def by(frame, col):
        return {k: int(v) for k, v in frame[col].value_counts().items()}

    meta = {
        "dataset": "Coveo SIGIR eCom 2021", "T_MAX": t_max, "LABEL_MAP": LABEL_MAP, "CLASS_ORDER": CLASS_ORDER,
        "session_cat_cols": SESSION_CAT_COLS, "key_events": KEY_EVENTS, "item_relations": ITEM_RELATIONS,
        "split_boundaries": {"p70": p70, "p85": p85},
        "dt_stats": {"dt_mean_sec": dt_mean, "dt_std_sec": dt_std, "MAX_DT_sec": max_dt},
        "price_stats": {"price_bucket_mean": p_mean, "price_bucket_std": p_std},
        "vocab_sizes": {"event": len(event_vocab), "page": len(page_vocab), "item": len(item_vocab)},
        "event_vocab": event_vocab, "cat_vocab_sizes": {c: len(v) for c, v in cat_vocabs.items()},
        "sample": info,
        "counts": {"sessions": int(len(sess)), "sessions_by_split": by(sess, "split"),
                   "sessions_by_class": by(sess, "y"), "prefixes": int(n),
                   "prefixes_by_split": {s: int((split_arr == s).sum()) for s in ("train", "val", "test")},
                   "test_prefixes_by_class": {c: int(((split_arr == "test") & (y_arr == LABEL_MAP[c])).sum())
                                              for c in CLASS_ORDER},
                   "undetermined_test_prefixes": int(undetermined[split_arr == "test"].sum())},
        "repro_all_pass": None, "leakage": None,
    }
    with open(os.path.join(out_dir, "meta.json"), "w") as f:
        json.dump(meta, f, indent=2, default=int)
    with open(os.path.join(out_dir, "cat_vocabs.json"), "w") as f:
        json.dump(cat_vocabs, f)
    print(json.dumps(meta["counts"], indent=2))
    return meta


In [ ]:
%%writefile /content/exp4lib/batching.py
"""Numpy batch builders. A batch is a set of prefix rows; prefix row r is the
prefix that ends at event row r, i.e. events sess_start[s] .. r."""
import numpy as np
import torch


def _excl_cumsum(a):
    out = np.zeros_like(a)
    np.cumsum(a[:-1], out=out[1:])
    return out


def _ranges(starts, counts):
    """Concatenate arange(s, s+c) for every (s, c)."""
    total = int(counts.sum())
    if total == 0:
        return np.zeros(0, dtype=np.int64)
    return np.repeat(starts - _excl_cumsum(counts), counts) + np.arange(total)


def _session_tensors(D, rows, join, device):
    sess = D.ev_sess[rows]
    return {
        "cat": torch.from_numpy(D.sess_cat[sess]).to(device),
        "num": torch.from_numpy(D.session_numeric(rows, join)).to(device),
        "y": torch.from_numpy(D.y[rows]).to(device),
    }


def _prefix_events(D, rows):
    t = D.prefix_len[rows]
    start = D.sess_start[D.ev_sess[rows]]
    erows = _ranges(start, t)
    owner = np.repeat(np.arange(len(rows)), t)
    offs = np.arange(len(erows)) - np.repeat(_excl_cumsum(t), t)
    return t, erows, owner, offs


def _item_edges(D, erows, join):
    ptr, item, rel = D.edges[join]
    cnt = ptr[erows + 1] - ptr[erows]
    eidx = _ranges(ptr[erows], cnt)
    src = np.repeat(np.arange(len(erows)), cnt)  # index into erows
    return src, item[eidx], rel[eidx]


def graph_batch(D, rows, device, join="fixed", with_page_item=True):
    """Heterogeneous prefix graphs, identical in content to Exp3 build_prefix_subgraph.

    Event nodes: one per prefix event (event_name_idx, position 1..t, delta_t_prev_norm).
    Page/Item nodes: deduplicated within each prefix. Edges: next_event, on_page,
    view_item_of / add_to_cart_of / purchase_item_of (reverse edges added in the model).
    """
    t, erows, owner, offs = _prefix_events(D, rows)
    n_ev = len(erows)
    b = {
        "event_name": torch.from_numpy(D.ev_name[erows]).to(device),
        "event_num": torch.from_numpy(np.stack([(offs + 1).astype(np.float32),
                                                D.ev_dtn[erows]], axis=1)).to(device),
        "readout": torch.from_numpy(np.cumsum(t) - 1).to(device),
        "event_graph": torch.from_numpy(owner).to(device),  # prefix id of every event node (pooling)
    }
    nxt = np.flatnonzero(offs < np.repeat(t, t) - 1)
    b["next_event"] = torch.from_numpy(np.stack([nxt, nxt + 1])).to(device)
    b.update(_session_tensors(D, rows, join, device))
    if not with_page_item:
        return b

    P = int(D.vocab["page"])
    pkey = owner * P + D.ev_page[erows]
    upk, pinv = np.unique(pkey, return_inverse=True)
    b["page_idx"] = torch.from_numpy(upk % P).to(device)
    b["on_page"] = torch.from_numpy(np.stack([np.arange(n_ev), pinv.reshape(-1)])).to(device)

    src, item, rel = _item_edges(D, erows, join)
    I = int(D.vocab["item"])
    ikey = owner[src] * I + item
    uik, iinv = np.unique(ikey, return_inverse=True)
    item_idx = uik % I
    iinv = iinv.reshape(-1)
    b["item_idx"] = torch.from_numpy(item_idx).to(device)
    b["item_price"] = torch.from_numpy(D.price_lookup[item_idx]).to(device)
    for r, name in enumerate(D.item_relations):
        m = rel == r
        b[name] = torch.from_numpy(np.stack([src[m], iinv[m]])).to(device)
    return b


def event_feature_batch(D, rows, device, join="fixed"):
    """Event chain with page/item as node features (homogeneous GraphSAGE)."""
    t, erows, owner, offs = _prefix_events(D, rows)
    b = {
        "event_name": torch.from_numpy(D.ev_name[erows]).to(device),
        "event_num": torch.from_numpy(np.stack([(offs + 1).astype(np.float32),
                                                D.ev_dtn[erows]], axis=1)).to(device),
        "page_idx": torch.from_numpy(D.ev_page[erows]).to(device),
        "readout": torch.from_numpy(np.cumsum(t) - 1).to(device),
        "event_graph": torch.from_numpy(owner).to(device),
    }
    nxt = np.flatnonzero(offs < np.repeat(t, t) - 1)
    b["next_event"] = torch.from_numpy(np.stack([nxt, nxt + 1])).to(device)
    src, item, _ = _item_edges(D, erows, join)
    b["item_tok"] = torch.from_numpy(src).to(device)
    b["item_idx"] = torch.from_numpy(item).to(device)
    b["item_price"] = torch.from_numpy(D.price_lookup[item]).to(device)
    b["n_tok"] = len(erows)
    b.update(_session_tensors(D, rows, join, device))
    return b


def seq_batch(D, rows, device, join="fixed"):
    """Right-padded sequences (length = longest prefix in the batch).

    Right padding + causal attention is equivalent to the Exp2 left-padded SASRec
    (positions 0..t-1, readout at the last real token) and avoids fully-masked rows.
    """
    t, erows, owner, offs = _prefix_events(D, rows)
    B, L = len(rows), int(t.max())
    flat = owner * L + offs
    tok = np.zeros(B * L, dtype=np.int64); tok[flat] = D.ev_name[erows]
    page = np.zeros(B * L, dtype=np.int64); page[flat] = D.ev_page[erows]
    dt = np.zeros(B * L, dtype=np.float32); dt[flat] = D.ev_dt01[erows]
    src, item, _ = _item_edges(D, erows, join)
    b = {
        "tok": torch.from_numpy(tok.reshape(B, L)).to(device),
        "page": torch.from_numpy(page.reshape(B, L)).to(device),
        "dt": torch.from_numpy(dt.reshape(B, L)).to(device),
        "len": torch.from_numpy(t).to(device),
        "item_tok": torch.from_numpy(flat[src]).to(device),
        "item_idx": torch.from_numpy(item).to(device),
        "item_price": torch.from_numpy(D.price_lookup[item]).to(device),
    }
    b.update(_session_tensors(D, rows, join, device))
    return b


In [ ]:
%%writefile /content/exp4lib/models.py
"""Exp4 neural models. Graph models follow Exp3 Section 6 / 10.4, sequence models
follow Exp2 Section 6.6. All take the dict batches from batching.py."""
import functools

import torch
import torch.nn as nn
from torch_geometric.nn import HeteroConv, SAGEConv
from torch_geometric.utils import scatter, softmax

from . import batching

DEFAULT_ITEM_RELATIONS = ["view_item_of", "add_to_cart_of", "purchase_item_of"]  # Exp3/Exp4


def hetero_edge_types(item_relations=DEFAULT_ITEM_RELATIONS):
    """Edge types in the Exp3 order (this order fixes parameter initialisation)."""
    types = [("event", "next_event", "event"),
             ("event", "on_page", "page"), ("page", "rev_on_page", "event")]
    for rel in item_relations:
        types += [("event", rel, "item"), ("item", "rev_" + rel, "event")]
    return types


HETERO_EDGE_TYPES = hetero_edge_types()


class SessionVector(nn.Module):
    """Exp3 session features: one hidden_dim embedding per categorical + raw numerics."""

    def __init__(self, cat_vocab_sizes, n_numeric, hidden_dim):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(s + 1, hidden_dim) for s in cat_vocab_sizes])
        self.dim = len(cat_vocab_sizes) * hidden_dim + n_numeric

    def forward(self, b):
        return torch.cat([e(b["cat"][:, i]) for i, e in enumerate(self.embs)] + [b["num"]], dim=-1)


def mlp_head(in_dim, hidden_dim, n_classes, dropout):
    return nn.Sequential(nn.Linear(in_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
                         nn.Linear(hidden_dim, n_classes))


class ItemTokenVector(nn.Module):
    """Mean over a token's items of (item embedding + price projection); zero if none."""

    def __init__(self, item_vocab, hidden_dim):
        super().__init__()
        self.emb = nn.Embedding(item_vocab, hidden_dim, padding_idx=0)
        self.price = nn.Linear(1, hidden_dim)

    def forward(self, b, n_tok):
        v = self.emb(b["item_idx"]) + self.price(b["item_price"].unsqueeze(-1))
        out = torch.zeros(n_tok, v.shape[1], device=v.device, dtype=v.dtype)
        out.index_add_(0, b["item_tok"], v)
        cnt = torch.zeros(n_tok, device=v.device, dtype=v.dtype)
        cnt.index_add_(0, b["item_tok"], torch.ones_like(b["item_price"]))
        return out / cnt.clamp(min=1).unsqueeze(-1)


# --------------------------------------------------------------------------
# Graph models
# --------------------------------------------------------------------------
class SessionHGNN(nn.Module):
    """Exp3 SessionHGNN (HeteroConv + SAGEConv, last-event readout).

    Exp4 stage 3b options: n_layers (Exp3 = 2) and readout:
      "last" - Exp3: embedding of the last event node only
      "mean" - [last event, mean over all event nodes of the prefix]
      "attn" - [last event, gated attention pooling over all event nodes]
    """

    def __init__(self, vocab, cat_vocab_sizes, hidden_dim=64, n_classes=4, dropout=0.3,
                 n_layers=2, readout="last", item_relations=DEFAULT_ITEM_RELATIONS):
        super().__init__()
        H = hidden_dim
        self.H = H
        self.readout = readout
        self.item_relations = list(item_relations)
        edge_types = hetero_edge_types(self.item_relations)
        self.event_emb = nn.Embedding(vocab["event"], H, padding_idx=0)
        self.page_emb = nn.Embedding(vocab["page"], H, padding_idx=0)
        self.item_emb = nn.Embedding(vocab["item"], H, padding_idx=0)
        self.event_numeric_proj = nn.Linear(2, H)
        self.item_numeric_proj = nn.Linear(1, H)
        self.convs = nn.ModuleList([
            HeteroConv({et: SAGEConv((H, H), H) for et in edge_types}, aggr="sum")
            for _ in range(n_layers)])
        if readout == "attn":
            self.gate = nn.Sequential(nn.Linear(H, H), nn.Tanh(), nn.Linear(H, 1))
        graph_dim = H if readout == "last" else 2 * H
        self.session = SessionVector(cat_vocab_sizes, 4, H)
        self.head = mlp_head(graph_dim + self.session.dim, H, n_classes, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, b):
        x_event = self.event_emb(b["event_name"]) + self.event_numeric_proj(b["event_num"])
        x_page = self.page_emb(b["page_idx"])
        x_item = self.item_emb(b["item_idx"]) + self.item_numeric_proj(b["item_price"].unsqueeze(-1))
        x = {"event": x_event, "page": x_page, "item": x_item}
        ei = {("event", "next_event", "event"): b["next_event"]}
        for rel, dst in [("on_page", "page")] + [(r, "item") for r in self.item_relations]:
            ei[("event", rel, dst)] = b[rel]
            ei[(dst, "rev_" + rel, "event")] = b[rel].flip(0)
        for conv in self.convs:
            x = conv(x, ei)
            x = {k: self.dropout(v.relu()) for k, v in x.items()}
        xe = x["event"]
        h = xe[b["readout"]]
        if self.readout != "last":
            g, B = b["event_graph"], h.shape[0]
            if self.readout == "mean":
                pooled = scatter(xe, g, dim=0, dim_size=B, reduce="mean")
            else:
                a = softmax(self.gate(xe), g, num_nodes=B)
                pooled = scatter(a * xe, g, dim=0, dim_size=B, reduce="sum")
            h = torch.cat([h, pooled], dim=-1)
        return self.head(torch.cat([h, self.session(b)], dim=-1))


class EventOnlyHGNN(nn.Module):
    """Exp3 10.4 ablation: Event nodes + next_event only. use_session=False removes
    the session vector from the head entirely (new Exp4 variant)."""

    def __init__(self, vocab, cat_vocab_sizes, hidden_dim=64, n_classes=4, dropout=0.3, use_session=True):
        super().__init__()
        H = hidden_dim
        et = ("event", "next_event", "event")
        self.event_emb = nn.Embedding(vocab["event"], H, padding_idx=0)
        self.event_numeric_proj = nn.Linear(2, H)
        self.conv1 = HeteroConv({et: SAGEConv((H, H), H)}, aggr="sum")
        self.conv2 = HeteroConv({et: SAGEConv((H, H), H)}, aggr="sum")
        self.use_session = use_session
        self.session = SessionVector(cat_vocab_sizes, 4, H) if use_session else None
        self.head = mlp_head(H + (self.session.dim if use_session else 0), H, n_classes, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, b):
        x = {"event": self.event_emb(b["event_name"]) + self.event_numeric_proj(b["event_num"])}
        ei = {("event", "next_event", "event"): b["next_event"]}
        x = self.conv1(x, ei)
        x = {k: self.dropout(v.relu()) for k, v in x.items()}
        x = self.conv2(x, ei)
        x = {k: self.dropout(v.relu()) for k, v in x.items()}
        h = x["event"][b["readout"]]
        if self.use_session:
            h = torch.cat([h, self.session(b)], dim=-1)
        return self.head(h)


class HomoGraphSAGE(nn.Module):
    """Homogeneous GraphSAGE on the event chain. Page and item identity enter as
    node features of the event (no shared Page/Item nodes, no extra relations).
    n_layers / readout as in SessionHGNN (stage 3c: pooled-readout control)."""

    def __init__(self, vocab, cat_vocab_sizes, hidden_dim=64, n_classes=4, dropout=0.3,
                 n_layers=2, readout="last"):
        super().__init__()
        H = hidden_dim
        self.readout = readout
        self.event_emb = nn.Embedding(vocab["event"], H, padding_idx=0)
        self.event_numeric_proj = nn.Linear(2, H)
        self.page_emb = nn.Embedding(vocab["page"], H, padding_idx=0)
        self.items = ItemTokenVector(vocab["item"], H)
        self.convs = nn.ModuleList([SAGEConv(H, H) for _ in range(n_layers)])
        self.session = SessionVector(cat_vocab_sizes, 4, H)
        graph_dim = H if readout == "last" else 2 * H
        self.head = mlp_head(graph_dim + self.session.dim, H, n_classes, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, b):
        x = (self.event_emb(b["event_name"]) + self.event_numeric_proj(b["event_num"])
             + self.page_emb(b["page_idx"]) + self.items(b, b["n_tok"]))
        for conv in self.convs:
            x = self.dropout(conv(x, b["next_event"]).relu())
        h = x[b["readout"]]
        if self.readout == "mean":
            pooled = scatter(x, b["event_graph"], dim=0, dim_size=h.shape[0], reduce="mean")
            h = torch.cat([h, pooled], dim=-1)
        return self.head(torch.cat([h, self.session(b)], dim=-1))


# --------------------------------------------------------------------------
# Sequence models
# --------------------------------------------------------------------------
class SeqInput(nn.Module):
    """Per-token input: event + time (+ page + items)."""

    def __init__(self, vocab, H, use_page_item, use_pos, max_len=43):  # max_len = T_MAX
        super().__init__()
        self.token_emb = nn.Embedding(vocab["event"], H, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len, H) if use_pos else None
        self.time_proj = nn.Linear(1, H)
        self.use_page_item = use_page_item
        if use_page_item:
            self.page_emb = nn.Embedding(vocab["page"], H, padding_idx=0)
            self.items = ItemTokenVector(vocab["item"], H)

    def forward(self, b):
        B, L = b["tok"].shape
        x = self.token_emb(b["tok"]) + self.time_proj(b["dt"].unsqueeze(-1))
        if self.pos_emb is not None:
            x = x + self.pos_emb(torch.arange(L, device=x.device)).unsqueeze(0)
        if self.use_page_item:
            x = x + self.page_emb(b["page"]) + self.items(b, B * L).view(B, L, -1)
        return x


class SASRec(nn.Module):
    """Exp2 SASRec+Time (2 layers, 2 heads, causal attention, last-token readout).
    use_session adds the Exp3 session vector before an MLP head (as in the HGNN);
    use_page_item adds page and item embeddings to every event token."""

    def __init__(self, vocab, cat_vocab_sizes, hidden_dim=64, n_classes=4, dropout=0.1,
                 num_heads=2, num_layers=2, use_session=False, use_page_item=False, max_len=43):
        super().__init__()
        H = hidden_dim
        self.inp = SeqInput(vocab, H, use_page_item, use_pos=True, max_len=max_len)
        self.layer_norm = nn.LayerNorm(H)
        self.dropout = nn.Dropout(dropout)
        layer = nn.TransformerEncoderLayer(d_model=H, nhead=num_heads, dim_feedforward=4 * H,
                                           dropout=dropout, activation="gelu", batch_first=True)
        self.transformer = nn.TransformerEncoder(layer, num_layers=num_layers, enable_nested_tensor=False)
        self.use_session = use_session
        if use_session:
            self.session = SessionVector(cat_vocab_sizes, 4, H)
            self.head = mlp_head(H + self.session.dim, H, n_classes, dropout)
        else:
            self.head = nn.Linear(H, n_classes)

    def forward(self, b):
        x = self.dropout(self.layer_norm(self.inp(b)))
        L = x.shape[1]
        causal = torch.triu(torch.ones(L, L, device=x.device, dtype=torch.bool), diagonal=1)
        x = self.transformer(x, mask=causal)
        h = x[torch.arange(x.shape[0], device=x.device), b["len"] - 1]
        if self.use_session:
            h = torch.cat([h, self.session(b)], dim=-1)
        return self.head(h)


class GRUClassifier(nn.Module):
    """GRU4Rec-style encoder with the same inputs as SASRec+Time+SF+PageItem."""

    def __init__(self, vocab, cat_vocab_sizes, hidden_dim=64, n_classes=4, dropout=0.1):
        super().__init__()
        H = hidden_dim
        self.inp = SeqInput(vocab, H, use_page_item=True, use_pos=False)
        self.dropout = nn.Dropout(dropout)
        self.gru = nn.GRU(H, H, num_layers=1, batch_first=True)
        self.session = SessionVector(cat_vocab_sizes, 4, H)
        self.head = mlp_head(H + self.session.dim, H, n_classes, dropout)

    def forward(self, b):
        x, _ = self.gru(self.dropout(self.inp(b)))  # causal: output at t-1 sees tokens < t only
        h = x[torch.arange(x.shape[0], device=x.device), b["len"] - 1]
        return self.head(torch.cat([h, self.session(b)], dim=-1))


# --------------------------------------------------------------------------
# Registry. Training settings are those of the thesis for each family:
# graph models = Exp3 7.0, sequence models = Exp2 6.5.
# --------------------------------------------------------------------------
GRAPH_TRAIN = {"batch_size": 256, "lr": 1e-3, "max_epochs": 50, "patience": 7}
SEQ_TRAIN = {"batch_size": 512, "lr": 1e-3, "max_epochs": 30, "patience": 3}


def _pb(builder, **kw):
    return functools.partial(builder, **kw)


MODEL_SPECS = {
    "SASRec+Time": dict(
        make=lambda D: SASRec(D.vocab, D.cat_vocab_sizes, max_len=D.t_max),
        batch=_pb(batching.seq_batch), train=SEQ_TRAIN),
    "SASRec+Time+SF": dict(
        make=lambda D: SASRec(D.vocab, D.cat_vocab_sizes, use_session=True, max_len=D.t_max),
        batch=_pb(batching.seq_batch), train=SEQ_TRAIN),
    "SASRec+Time+SF+PageItem": dict(
        make=lambda D: SASRec(D.vocab, D.cat_vocab_sizes, use_session=True, use_page_item=True,
                              max_len=D.t_max),
        batch=_pb(batching.seq_batch), train=SEQ_TRAIN),
    "GRU+Time+SF+PageItem": dict(
        make=lambda D: GRUClassifier(D.vocab, D.cat_vocab_sizes),
        batch=_pb(batching.seq_batch), train=SEQ_TRAIN),
    "HGNN_full": dict(
        make=lambda D: SessionHGNN(D.vocab, D.cat_vocab_sizes, item_relations=D.item_relations),
        batch=_pb(batching.graph_batch), train=GRAPH_TRAIN),
    "HGNN_no_page_item": dict(
        make=lambda D: EventOnlyHGNN(D.vocab, D.cat_vocab_sizes),
        batch=_pb(batching.graph_batch, with_page_item=False), train=GRAPH_TRAIN),
    "HGNN_no_page_item_no_SF": dict(
        make=lambda D: EventOnlyHGNN(D.vocab, D.cat_vocab_sizes, use_session=False),
        batch=_pb(batching.graph_batch, with_page_item=False), train=GRAPH_TRAIN),
    "HomoGraphSAGE+PageItemFeat": dict(
        make=lambda D: HomoGraphSAGE(D.vocab, D.cat_vocab_sizes),
        batch=_pb(batching.event_feature_batch), train=GRAPH_TRAIN),
    # Stage 3b: readout / depth variants of HGNN_full (same protocol, fixed join)
    "HGNN_full_meanpool": dict(
        make=lambda D: SessionHGNN(D.vocab, D.cat_vocab_sizes, readout="mean",
                                   item_relations=D.item_relations),
        batch=_pb(batching.graph_batch), train=GRAPH_TRAIN),
    "HGNN_full_attnpool": dict(
        make=lambda D: SessionHGNN(D.vocab, D.cat_vocab_sizes, readout="attn",
                                   item_relations=D.item_relations),
        batch=_pb(batching.graph_batch), train=GRAPH_TRAIN),
    "HGNN_full_3layer_meanpool": dict(
        make=lambda D: SessionHGNN(D.vocab, D.cat_vocab_sizes, n_layers=3, readout="mean",
                                   item_relations=D.item_relations),
        batch=_pb(batching.graph_batch), train=GRAPH_TRAIN),
    # Stage 3c: homogeneous counterparts of the pooled HGNNs (heterogeneity control)
    "HomoGraphSAGE_meanpool": dict(
        make=lambda D: HomoGraphSAGE(D.vocab, D.cat_vocab_sizes, readout="mean"),
        batch=_pb(batching.event_feature_batch), train=GRAPH_TRAIN),
    "HomoGraphSAGE_3layer_meanpool": dict(
        make=lambda D: HomoGraphSAGE(D.vocab, D.cat_vocab_sizes, n_layers=3, readout="mean"),
        batch=_pb(batching.event_feature_batch), train=GRAPH_TRAIN),
    # Reference only: HGNN full trained/evaluated on the original Exp3 item join
    "HGNN_full_exp3join": dict(
        make=lambda D: SessionHGNN(D.vocab, D.cat_vocab_sizes, item_relations=D.item_relations),
        batch=_pb(batching.graph_batch, join="exp3"), train=GRAPH_TRAIN),
}


In [ ]:
%%writefile /content/exp4lib/train.py
"""Resumable multi-seed training with early stopping on val Macro-F1.

For every (model, seed) the following files live in <run_dir>:
  ckpt/<model>__seed<seed>.pt      epoch-level checkpoint (deleted when the run is done)
  preds/<model>__seed<seed>.parquet test predictions + class probabilities
  runs/<model>__seed<seed>.json    history, best epoch, val/test Macro-F1, runtime
"""
import json
import os
import random
import shutil
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import f1_score

from .models import MODEL_SPECS

EVAL_BATCH = 2048


def set_global_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def run_tag(model_name, seed):
    return f"{model_name}__seed{seed}"


def is_done(run_dir, model_name, seed):
    return os.path.exists(os.path.join(run_dir, "runs", run_tag(model_name, seed) + ".json"))


@torch.no_grad()
def predict(model, D, rows, build, device, batch_size=EVAL_BATCH):
    model.eval()
    out = []
    for i in range(0, len(rows), batch_size):
        b = build(D, rows[i:i + batch_size], device)
        out.append(torch.softmax(model(b).float(), dim=-1).cpu())
    return torch.cat(out).numpy()


def macro_f1(y, pred):
    return float(f1_score(y, pred, average="macro", labels=[0, 1, 2, 3], zero_division=0))


def _as_rng_state(s):
    """Saved RNG state (tensor on any device, numpy array, list, bytes) -> CPU ByteTensor."""
    if isinstance(s, (bytes, bytearray)):
        s = list(s)
    if isinstance(s, torch.Tensor):
        return s.detach().to(device="cpu", dtype=torch.uint8).contiguous()
    return torch.as_tensor(np.asarray(s)).to(device="cpu", dtype=torch.uint8).contiguous()


def _restore_rng(ck, log, tag):
    """Best effort only: data order comes from (seed, epoch), so a failed restore merely
    changes dropout noise after the resume point. Never fatal."""
    try:
        torch.set_rng_state(_as_rng_state(ck["torch_rng"]))
    except Exception as e:
        log(f"[{tag}] warning: CPU RNG state not restored ({type(e).__name__}: {e})")
    states = ck.get("cuda_rng")
    if not torch.cuda.is_available() or not states:
        return
    try:
        states = [_as_rng_state(s) for s in states][:torch.cuda.device_count()]
        torch.cuda.set_rng_state_all(states)  # module attribute, looked up at call time
    except Exception as e:
        log(f"[{tag}] warning: CUDA RNG state not restored ({type(e).__name__}: {e})")


def _atomic_save(obj, path):
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)


def train_one(D, model_name, seed, run_dir, device, log=print, max_train_rows=None, lr=None):
    """Train one (model, seed). Resumes from the epoch checkpoint if present.
    lr overrides the family learning rate (learning-rate selection stage)."""
    spec = MODEL_SPECS[model_name]
    cfg = dict(spec["train"], lr=lr) if lr is not None else spec["train"]
    build = spec["batch"]
    tag = run_tag(model_name, seed)
    for sub in ("ckpt", "preds", "runs"):
        os.makedirs(os.path.join(run_dir, sub), exist_ok=True)
    ckpt_path = os.path.join(run_dir, "ckpt", tag + ".pt")
    if is_done(run_dir, model_name, seed):
        log(f"[{tag}] already done, skipping")
        return

    set_global_seed(seed)
    model = spec["make"](D).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
    crit = nn.CrossEntropyLoss(weight=torch.from_numpy(D.class_weights()).to(device))

    train_rows = D.rows["train"]
    if max_train_rows is not None:  # smoke tests only
        train_rows = train_rows[:max_train_rows]
    val_rows = D.rows["val"]

    state = {"epoch": 0, "best_val": -1.0, "best_epoch": 0, "patience": 0,
             "history": [], "best_state": None, "elapsed": 0.0}
    if os.path.exists(ckpt_path):
        # load on CPU: RNG states must be CPU ByteTensors; model/optimizer move to device on load
        ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        model.load_state_dict(ck["model"])
        opt.load_state_dict(ck["opt"])
        state = ck["state"]
        _restore_rng(ck, log, tag)
        log(f"[{tag}] resumed after epoch {state['epoch']}")

    while state["epoch"] < cfg["max_epochs"] and state["patience"] < cfg["patience"]:
        epoch = state["epoch"] + 1
        t0 = time.time()
        model.train()
        perm = np.random.default_rng([seed, epoch]).permutation(train_rows)
        bs = cfg["batch_size"]
        loss_sum = 0.0
        for i in range(0, len(perm), bs):
            rows = perm[i:i + bs]
            b = build(D, rows, device)
            opt.zero_grad(set_to_none=True)
            loss = crit(model(b), b["y"])
            loss.backward()
            opt.step()
            loss_sum += float(loss.detach()) * len(rows)
        val_prob = predict(model, D, val_rows, build, device)
        val_f1 = macro_f1(D.y[val_rows], val_prob.argmax(1))
        dt = time.time() - t0
        state["elapsed"] += dt
        state["history"].append({"epoch": epoch, "train_loss": loss_sum / len(perm),
                                 "val_macro_f1": val_f1, "seconds": round(dt, 1)})
        if val_f1 > state["best_val"]:
            state.update(best_val=val_f1, best_epoch=epoch, patience=0,
                         best_state={k: v.detach().cpu().clone() for k, v in model.state_dict().items()})
            mark = " *"
        else:
            state["patience"] += 1
            mark = ""
        state["epoch"] = epoch
        log(f"[{tag}] epoch {epoch:2d} loss={loss_sum / len(perm):.4f} val_f1={val_f1:.4f}{mark} ({dt:.0f}s)")
        _atomic_save({"model": model.state_dict(), "opt": opt.state_dict(), "state": state,
                      "torch_rng": torch.get_rng_state(),
                      "cuda_rng": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None},
                     ckpt_path)

    model.load_state_dict(state["best_state"])
    t0 = time.time()
    test_rows = D.rows["test"]
    prob = predict(model, D, test_rows, build, device)
    pred = prob.argmax(1)
    test_f1 = macro_f1(D.y[test_rows], pred)
    state["elapsed"] += time.time() - t0

    df = pd.DataFrame({"row": test_rows, "pred": pred.astype(np.int8)})
    for c in range(4):
        df[f"p{c}"] = prob[:, c].astype(np.float32)
    df.to_parquet(os.path.join(run_dir, "preds", tag + ".parquet"), index=False)
    torch.save(state["best_state"], os.path.join(run_dir, "ckpt", tag + "__best.pt"))
    info = {"model": model_name, "seed": seed, "best_epoch": state["best_epoch"],
            "best_val_macro_f1": state["best_val"], "test_macro_f1": test_f1,
            "epochs_run": state["epoch"], "runtime_sec": round(state["elapsed"], 1),
            "n_params": sum(p.numel() for p in model.parameters()),
            "train_config": cfg, "history": state["history"]}
    with open(os.path.join(run_dir, "runs", tag + ".json"), "w") as f:
        json.dump(info, f, indent=2)
    if os.path.exists(ckpt_path):
        os.remove(ckpt_path)
    log(f"[{tag}] done: best_epoch={state['best_epoch']} val={state['best_val']:.4f} "
        f"test={test_f1:.4f} ({state['elapsed'] / 60:.1f} min)")
    return info


# --------------------------------------------------------------------------
# Learning-rate selection
# --------------------------------------------------------------------------
LR_GRID = (1e-3, 3e-4, 1e-4)


def _run_info(run_dir, model_name, seed):
    with open(os.path.join(run_dir, "runs", run_tag(model_name, seed) + ".json")) as f:
        return json.load(f)


def _copy_run(src_dir, dst_dir, tag):
    for sub, ext in (("runs", ".json"), ("preds", ".parquet")):
        os.makedirs(os.path.join(dst_dir, sub), exist_ok=True)
        shutil.copy2(os.path.join(src_dir, sub, tag + ext), os.path.join(dst_dir, sub, tag + ext))


def reuse_from_lr_final(base_run_dir, lr_root, model_name, seed, log=print):
    """Fixed-rate stages: if stage 5 already trained this (model, seed) at the default
    learning rate (its chosen rate equals the default), copy that run instead of retraining."""
    choice_path = os.path.join(lr_root, "lr_choice.json")
    final_dir = os.path.join(lr_root, "final")
    if model_name not in MODEL_SPECS or not os.path.exists(choice_path):
        return False
    chosen = json.load(open(choice_path)).get(model_name, {}).get("chosen_lr")
    default_lr = MODEL_SPECS[model_name]["train"]["lr"]
    if chosen is None or abs(chosen - default_lr) > 1e-12 or not is_done(final_dir, model_name, seed):
        return False
    _copy_run(final_dir, base_run_dir, run_tag(model_name, seed))
    log(f"[{run_tag(model_name, seed)}] reused from lr_selection/final (chosen lr = default {default_lr:g})")
    return True


def lr_selection(D, models, base_run_dir, lr_root, device, lrs=LR_GRID, select_seed=42,
                 seeds=(42, 123, 456), copy_prefixes=("Markov-", "LightGBM__"), log=print):
    """Choose each neural model's learning rate on validation Macro-F1 (one seed), then
    train all seeds at the chosen rate. Everything else (patience, batch size, epochs,
    class weights) is unchanged.

    Layout under lr_root:
      search/lr<value>/   select_seed runs for every learning rate
      final/              all seeds at the chosen rate (+ copied non-neural baselines)
      lr_choice.json      validation scores and the chosen rate per model
    Runs that already exist are reused: the family default rate for select_seed and
    for all seeds comes from base_run_dir, so lr=1e-3 is never retrained.
    """
    search_root = os.path.join(lr_root, "search")
    final_dir = os.path.join(lr_root, "final")
    choice_path = os.path.join(lr_root, "lr_choice.json")
    os.makedirs(final_dir, exist_ok=True)
    choice = json.load(open(choice_path)) if os.path.exists(choice_path) else {}

    for model in models:
        default_lr = MODEL_SPECS[model]["train"]["lr"]

        def run_dir_for(lr_value):
            if abs(lr_value - default_lr) < 1e-12:
                return base_run_dir
            return os.path.join(search_root, f"lr{lr_value:g}")

        scores = {}
        for lr_value in lrs:
            d = run_dir_for(lr_value)
            if not is_done(d, model, select_seed):
                log(f"[lr-search] {model} lr={lr_value:g} seed={select_seed}")
                train_one(D, model, select_seed, d, device, log=log, lr=lr_value)
            scores[f"{lr_value:g}"] = _run_info(d, model, select_seed)["best_val_macro_f1"]
        best = max(lrs, key=lambda v: scores[f"{v:g}"])  # first best in grid order
        choice[model] = {"val_macro_f1_by_lr": scores, "chosen_lr": best, "select_seed": select_seed}
        with open(choice_path, "w") as f:
            json.dump(choice, f, indent=2)
        log(f"[lr-search] {model}: {scores} -> lr={best:g}")

        src = run_dir_for(best)
        for seed in seeds:
            tag = run_tag(model, seed)
            if is_done(final_dir, model, seed):
                continue
            if is_done(src, model, seed):
                _copy_run(src, final_dir, tag)
            else:
                train_one(D, model, seed, final_dir, device, log=log, lr=best)

    for fn in sorted(os.listdir(os.path.join(base_run_dir, "runs"))):
        if fn.startswith(copy_prefixes) and fn.endswith(".json") and "__seed" in fn:
            tag = fn[:-5]
            if not os.path.exists(os.path.join(final_dir, "runs", fn)):
                _copy_run(base_run_dir, final_dir, tag)
    return choice


In [ ]:
%%writefile /content/exp4lib/baselines.py
"""Markov-3 with stepwise backoff (Exp2 Section 4) and LightGBM on Exp2-style
prefix features plus the Exp3 session features (Exp2 Section 5), both on the
Exp3 dataset. Prefix row i == event row i (see data.py)."""
import json
import os
import time
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score


EXP2_LABEL_ORDER = [3, 2, 1, 0]  # Exp2 LABELS = Buyer, Intent, Researcher, Browser (argmax tie order)
ALPHA_GRID = [0.0, 0.05, 0.1, 0.2, 0.3, 0.5]


def _mf1(y, p):
    return float(f1_score(y, p, average="macro", labels=[0, 1, 2, 3], zero_division=0))


def _save_preds(path, rows, prob):
    df = pd.DataFrame({"row": rows, "pred": prob.argmax(1).astype(np.int8)})
    for c in range(4):
        df[f"p{c}"] = prob[:, c].astype(np.float32)
    df.to_parquet(path, index=False)


# --------------------------------------------------------------------------
# Markov-3
# --------------------------------------------------------------------------
def run_markov(D, sessions_df, run_dir, n=3, log=print):
    tag = f"Markov-{n}__seed0"
    if os.path.exists(os.path.join(run_dir, "runs", tag + ".json")):
        log(f"[{tag}] already done, skipping")
        return
    t0 = time.time()
    tok = D.ev_name
    pos = D.ev_pos
    codes = {}
    for k in range(1, n + 1):
        c = np.zeros(D.n, dtype=np.int64)
        for j in range(k):
            sh = np.r_[np.zeros(j, dtype=np.int64), tok[:D.n - j]] if j else tok
            c = c * 16 + sh
        c[pos < k] = -1  # state needs t >= k
        codes[k] = c

    tr = D.rows["train"]
    tables = {}
    for k in range(1, n + 1):
        m = codes[k][tr] >= 0
        df = pd.DataFrame({"s": codes[k][tr][m], "y": D.y[tr][m]})
        tab = df.groupby(["s", "y"]).size().unstack(fill_value=0).reindex(columns=range(4), fill_value=0)
        tables[k] = tab
    # prior from train session labels
    ss = sessions_df[sessions_df["split"] == "train"]["y"].map({"Browser": 0, "Researcher": 1, "Intent": 2, "Buyer": 3})
    prior = np.bincount(ss, minlength=4) / len(ss)

    def dists(rows):
        emp = np.full((len(rows), 4), np.nan)
        for k in range(1, n + 1):  # higher order overwrites lower when seen
            tab = tables[k]
            c = codes[k][rows]
            hit = np.isin(c, tab.index.to_numpy())
            cnt = tab.reindex(c[hit]).to_numpy(np.float64)
            emp[hit] = cnt / cnt.sum(1, keepdims=True)
        return emp

    def predict(emp, alpha):
        d = np.where(np.isnan(emp[:, :1]), prior[None, :], (1 - alpha) * np.nan_to_num(emp) + alpha * prior[None, :])
        ordered = d[:, EXP2_LABEL_ORDER]
        return d, np.array(EXP2_LABEL_ORDER)[ordered.argmax(1)]

    val = D.rows["val"]
    emp_val = dists(val)
    scores = {a: _mf1(D.y[val], predict(emp_val, a)[1]) for a in ALPHA_GRID}
    best_alpha = max(scores, key=scores.get)  # first best in grid order
    test = D.rows["test"]
    prob, pred = predict(dists(test), best_alpha)
    test_f1 = _mf1(D.y[test], pred)
    os.makedirs(os.path.join(run_dir, "preds"), exist_ok=True)
    os.makedirs(os.path.join(run_dir, "runs"), exist_ok=True)
    df = pd.DataFrame({"row": test, "pred": pred.astype(np.int8)})
    for c in range(4):
        df[f"p{c}"] = prob[:, c].astype(np.float32)
    df.to_parquet(os.path.join(run_dir, "preds", tag + ".parquet"), index=False)
    info = {"model": f"Markov-{n}", "seed": 0, "alpha_val_scores": scores, "best_alpha": best_alpha,
            "best_val_macro_f1": scores[best_alpha], "test_macro_f1": test_f1,
            "n_states": {k: int(len(t)) for k, t in tables.items()},
            "runtime_sec": round(time.time() - t0, 1)}
    with open(os.path.join(run_dir, "runs", tag + ".json"), "w") as f:
        json.dump(info, f, indent=2)
    log(f"[{tag}] alpha={best_alpha} val={scores[best_alpha]:.4f} test={test_f1:.4f}")


# --------------------------------------------------------------------------
# LightGBM features
# --------------------------------------------------------------------------
TOP_K = 50
NGRAM_COVERAGE = 0.95
CONTEXT_FIELDS_C = ["country", "region", "traffic_campaign"]


def _select_top_m(counter, coverage):
    total = sum(counter.values())
    out, cum = [], 0
    for ng, c in counter.most_common():
        cum += c
        out.append(ng)
        if cum >= total * coverage:
            break
    return out


def build_lgbm_features(D, events_df, sessions_df, out_path, log=print):
    """Exp2 Section 5.2 features computed for every prefix row at once."""
    if os.path.exists(out_path):
        log("LightGBM feature table exists, loading.")
        return pd.read_parquet(out_path)
    t0 = time.time()
    name = events_df["event_name"].to_numpy()
    s = events_df["session_idx"].to_numpy()
    ts_sec = events_df["event_timestamp"].to_numpy(np.int64) / 1e6
    tpos = D.prefix_len.astype(np.float32)
    vocab = sorted(pd.unique(name[D.rows["train"]]))
    F = {"prefix_len": tpos}

    grp = pd.Series(s)
    freqs = []
    for tokname in vocab:
        cnt = pd.Series((name == tokname).astype(np.int32)).groupby(grp).cumsum().to_numpy().astype(np.float32)
        F[f"count_{tokname}"] = cnt
        F[f"freq_{tokname}"] = cnt / tpos
        freqs.append(cnt / tpos)
    fr = np.stack(freqs, 1)
    with np.errstate(divide="ignore", invalid="ignore"):
        F["entropy"] = np.where(fr > 0, -fr * np.log(fr), 0).sum(1).astype(np.float32)
    for e in D.key_events:
        F[f"has_{e}"] = pd.Series((name == e).astype(np.int8)).groupby(grp).cummax().to_numpy().astype(np.float32)

    dt = events_df["dt_sec"].to_numpy(np.float64)
    first = D.prefix_len == 1
    posdt = np.where((~first) & (dt > 0), dt, 0.0)
    npos = pd.Series(((~first) & (dt > 0)).astype(np.int32)).groupby(grp).cumsum().to_numpy()
    sdt = pd.Series(posdt).groupby(grp).cumsum().to_numpy()
    F["avg_dt"] = np.where(npos > 0, sdt / np.maximum(npos, 1), 0.0).astype(np.float32)
    start_ts = pd.Series(ts_sec).groupby(grp).transform("first").to_numpy()
    F["time_since_start"] = (ts_sec - start_ts).astype(np.float32)
    max_dt = D.meta["dt_stats"]["MAX_DT_sec"]
    for e in D.key_events:
        last = pd.Series(np.where(name == e, ts_sec, np.nan)).groupby(grp).ffill().to_numpy()
        occ = ~np.isnan(last)
        F[f"has_{e}_occurred"] = occ.astype(np.float32)
        F[f"time_since_last_{e}"] = np.where(occ, ts_sec - np.nan_to_num(last), max_dt).astype(np.float32)

    # suffix n-grams, vocab from train (top-M covering 95% of occurrences)
    tok_id = pd.Series(name).map({v: i for i, v in enumerate(vocab)}).fillna(len(vocab)).to_numpy(np.int64)
    V = len(vocab) + 1
    prev1 = np.r_[-1, tok_id[:-1]]
    prev2 = np.r_[-1, -1, tok_id[:-2]]
    bg = np.where(D.prefix_len >= 2, prev1 * V + tok_id, -1)
    tg = np.where(D.prefix_len >= 3, (prev2 * V + prev1) * V + tok_id, -1)
    tr = D.rows["train"]
    for nm, codes in (("bigram", bg), ("trigram", tg)):
        c_tr = codes[tr]
        top = _select_top_m(Counter(c_tr[c_tr >= 0].tolist()), NGRAM_COVERAGE)
        idx = {c: i for i, c in enumerate(top)}
        mapped = pd.Series(codes).map(idx).fillna(len(top)).to_numpy(np.int64)
        mapped[codes < 0] = -1
        for i in range(len(top) + 1):  # last index = <OTHER_NGRAM>
            F[f"{nm}_{i}"] = (mapped == i).astype(np.uint8)

    # Exp3 session features
    F["unique_items"] = D.uniq["fixed"]
    F["cart_flag"] = D.cart
    F["is_new_user"] = D.sess_new[D.ev_sess]
    feats = pd.DataFrame(F)

    # context categoricals (Exp2 encoding: MISSING, top-K for high-cardinality, train-only)
    ss = sessions_df.set_index("session_idx")
    tr_sess = ss["split"].eq("train")
    for col in D.session_cat_cols:
        vals = ss["ctx_" + col].fillna("MISSING").astype(str)
        if col in CONTEXT_FIELDS_C:
            top = set(vals[tr_sess & (vals != "MISSING")].value_counts().iloc[:TOP_K].index) | {"MISSING"}
            vals = vals.where(vals.isin(top), "<OTHER>")
        known = sorted(set(vals[tr_sess]))
        fallback = "MISSING" if "MISSING" in known else vals[tr_sess].value_counts().idxmax()
        vals = vals.where(vals.isin(known), fallback)
        enc = {v: i for i, v in enumerate(known)}
        feats["ctx_" + col] = vals.map(enc).to_numpy(np.int32)[D.ev_sess]
    feats.to_parquet(out_path, index=False)
    log(f"LightGBM features: {feats.shape} in {time.time() - t0:.0f}s")
    return feats


def run_lightgbm(D, feats, sessions_df, run_dir, seeds, n_trials=30, tune_frac=1.0,
                 n_jobs=None, timeout_sec=None, log=print):
    import lightgbm as lgb
    import optuna

    n_jobs = n_jobs or os.cpu_count()
    cat_cols = ["ctx_" + c for c in D.session_cat_cols]
    cols = list(feats.columns)
    os.makedirs(os.path.join(run_dir, "runs"), exist_ok=True)
    os.makedirs(os.path.join(run_dir, "preds"), exist_ok=True)
    w_cls = D.class_weights()
    params_path = os.path.join(run_dir, "runs", "lgbm_best_params.json")

    if not os.path.exists(params_path):
        t0 = time.time()
        tr_s = sessions_df[sessions_df["split"] == "train"].sort_values("session_end_ts")
        cut = int(len(tr_s) * 0.8)
        fit_s = tr_s.iloc[:cut]["session_idx"].to_numpy()
        if tune_frac < 1.0:
            fit_s = np.random.default_rng(42).choice(fit_s, int(len(fit_s) * tune_frac), replace=False)
        hold_s = tr_s.iloc[cut:]["session_idx"].to_numpy()
        tr = D.rows["train"]
        fit_rows = tr[np.isin(D.ev_sess[tr], fit_s)]
        hold_rows = tr[np.isin(D.ev_sess[tr], hold_s)]
        Xf, yf, wf = feats.iloc[fit_rows], D.y[fit_rows], w_cls[D.y[fit_rows]]
        Xh, yh = feats.iloc[hold_rows], D.y[hold_rows]
        log(f"Optuna tuning: fit={len(fit_rows):,} holdout={len(hold_rows):,} prefixes, {n_trials} trials")

        def objective(trial):
            p = {
                "num_leaves": trial.suggest_categorical("num_leaves", [31, 63, 127, 255]),
                "max_depth": trial.suggest_categorical("max_depth", [-1, 8, 16]),
                "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.15, log=True),
                "n_estimators": trial.suggest_int("n_estimators", 200, 800),
                "min_child_samples": trial.suggest_categorical("min_child_samples", [20, 50, 100]),
                "subsample": trial.suggest_float("subsample", 0.7, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
                "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
            }
            m = lgb.LGBMClassifier(**p, objective="multiclass", random_state=42, n_jobs=n_jobs, verbose=-1)
            m.fit(Xf, yf, sample_weight=wf, categorical_feature=cat_cols, eval_set=[(Xh, yh)],
                  eval_metric="multi_logloss", callbacks=[lgb.early_stopping(30, verbose=False)])
            f = _mf1(yh, m.predict(Xh))
            log(f"  trial {trial.number}: macro_f1={f:.4f}")
            return f

        optuna.logging.set_verbosity(optuna.logging.WARNING)
        study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
        study.optimize(objective, n_trials=n_trials, n_jobs=1, timeout=timeout_sec)
        best = dict(study.best_params)
        with open(params_path, "w") as f:
            json.dump({"params": best, "tune_macro_f1": study.best_value,
                       "n_trials_completed": len(study.trials), "tune_frac": tune_frac,
                       "runtime_sec": round(time.time() - t0, 1)}, f, indent=2)
        del Xf, Xh
    with open(params_path) as f:
        tuned = json.load(f)
    best = tuned["params"]

    tr, va, te = D.rows["train"], D.rows["val"], D.rows["test"]
    for seed in seeds:
        tag = f"LightGBM__seed{seed}"
        if os.path.exists(os.path.join(run_dir, "runs", tag + ".json")):
            log(f"[{tag}] already done, skipping")
            continue
        t0 = time.time()
        m = lgb.LGBMClassifier(**best, objective="multiclass", random_state=seed, n_jobs=n_jobs, verbose=-1)
        m.fit(feats.iloc[tr], D.y[tr], sample_weight=w_cls[D.y[tr]], categorical_feature=cat_cols)
        val_f1 = _mf1(D.y[va], m.predict(feats.iloc[va]))
        prob = m.predict_proba(feats.iloc[te])
        test_f1 = _mf1(D.y[te], prob.argmax(1))
        _save_preds(os.path.join(run_dir, "preds", tag + ".parquet"), te, prob)
        info = {"model": "LightGBM", "seed": seed, "params": best, "best_val_macro_f1": val_f1,
                "test_macro_f1": test_f1, "n_features": len(cols),
                "tuning_runtime_sec": tuned["runtime_sec"], "runtime_sec": round(time.time() - t0, 1)}
        with open(os.path.join(run_dir, "runs", tag + ".json"), "w") as f:
            json.dump(info, f, indent=2)
        log(f"[{tag}] val={val_f1:.4f} test={test_f1:.4f} ({(time.time() - t0) / 60:.1f} min)")


In [ ]:
%%writefile /content/exp4lib/evaluate.py
"""Stage 4: metrics, paired session-level bootstrap, prefix-length bins,
undetermined subset, summary tables."""
import glob
import json
import os

import numpy as np
import pandas as pd

from .data import CLASS_ORDER

BINS = [(1, 1), (2, 2), (3, 3), (4, 4), (5, 5), (6, 7), (8, 10), (11, 15), (16, 20), (21, 10**6)]
BIN_LABELS = ["1", "2", "3", "4", "5", "6-7", "8-10", "11-15", "16-20", "21+"]
REFERENCE = "HGNN_full"
SECOND_REFERENCES = [  # extra paired-bootstrap baselines
    "GRU+Time+SF+PageItem",            # best sequence model (stage 3b question)
    "HomoGraphSAGE_3layer_meanpool",   # hetero vs homo under the pooled readout (stage 3c)
]


def f1_from_cm(cm):
    """Macro-F1 (and per-class F1) from confusion matrices [..., 4, 4] (rows=true)."""
    tp = np.diagonal(cm, axis1=-2, axis2=-1)
    fp = cm.sum(-2) - tp
    fn = cm.sum(-1) - tp
    den = 2 * tp + fp + fn
    f1 = np.where(den > 0, 2 * tp / np.maximum(den, 1e-12), 0.0)
    return f1.mean(-1), f1


def cm_of(y, p, mask=None):
    if mask is not None:
        y, p = y[mask], p[mask]
    return np.bincount(y * 4 + p, minlength=16).reshape(4, 4).astype(np.float64)


def load_runs(run_dir):
    """{model: {seed: (info, preds_df)}}"""
    out = {}
    for path in sorted(glob.glob(os.path.join(run_dir, "runs", "*__seed*.json"))):
        tag = os.path.basename(path)[:-5]
        pp = os.path.join(run_dir, "preds", tag + ".parquet")
        if not os.path.exists(pp):
            continue
        with open(path) as f:
            info = json.load(f)
        out.setdefault(info["model"], {})[int(info["seed"])] = (info, pd.read_parquet(pp))
    return out


def evaluate_all(D, run_dir, out_dir, n_boot=1000, boot_seed=2026, reference=REFERENCE,
                 extra_references=SECOND_REFERENCES, title=None, log=print):
    os.makedirs(out_dir, exist_ok=True)
    runs = load_runs(run_dir)
    if not runs:
        raise RuntimeError("No finished runs found.")
    test = D.rows["test"]
    y = D.y[test]
    plen = D.prefix_len[test]
    und = D.undetermined[test]
    sid_codes, sid_uni = pd.factorize(D.session_id[test])
    S = len(sid_uni)

    # ---- per model / per seed metrics -------------------------------------------------
    results = {}
    preds_by = {}
    for model, seeds in runs.items():
        per_seed = {}
        for seed, (info, df) in sorted(seeds.items()):
            assert np.array_equal(df["row"].to_numpy(), test), f"{model} seed {seed}: test rows differ"
            p = df["pred"].to_numpy(np.int64)
            preds_by[(model, seed)] = p
            mf, pc = f1_from_cm(cm_of(y, p))
            bins = {}
            for (lo, hi), lab in zip(BINS, BIN_LABELS):
                bins[lab] = f1_from_cm(cm_of(y, p, (plen >= lo) & (plen <= hi)))[0]
            per_seed[seed] = {
                "macro_f1": mf, "per_class_f1": dict(zip(CLASS_ORDER, pc.tolist())),
                "bins": bins, "undetermined_macro_f1": f1_from_cm(cm_of(y, p, und))[0],
                "determined_macro_f1": f1_from_cm(cm_of(y, p, ~und))[0],
                "best_val_macro_f1": info.get("best_val_macro_f1"),
                "best_epoch": info.get("best_epoch"), "runtime_sec": info.get("runtime_sec"),
            }
        mfs = np.array([v["macro_f1"] for v in per_seed.values()])
        mean = float(mfs.mean())
        rep = min(per_seed, key=lambda s: abs(per_seed[s]["macro_f1"] - mean))

        def agg(getter):
            vals = np.array([getter(v) for v in per_seed.values()], dtype=float)
            return {"mean": float(vals.mean()), "std": float(vals.std(ddof=1)) if len(vals) > 1 else 0.0}

        results[model] = {
            "n_seeds": len(per_seed), "seeds": sorted(per_seed), "representative_seed": int(rep),
            "macro_f1": agg(lambda v: v["macro_f1"]),
            "per_class_f1": {c: agg(lambda v, c=c: v["per_class_f1"][c]) for c in CLASS_ORDER},
            "bins": {b: agg(lambda v, b=b: v["bins"][b]) for b in BIN_LABELS},
            "undetermined_macro_f1": agg(lambda v: v["undetermined_macro_f1"]),
            "determined_macro_f1": agg(lambda v: v["determined_macro_f1"]),
            "runtime_sec_total": float(sum(v["runtime_sec"] or 0 for v in per_seed.values())),
            "per_seed": per_seed,
        }

    # ---- paired bootstrap over test sessions ------------------------------------------
    keys = list(preds_by)
    sess_cm = {}
    for k in keys:
        idx = sid_codes * 16 + y * 4 + preds_by[k]
        sess_cm[k] = np.bincount(idx, minlength=S * 16).reshape(S, 16).astype(np.float32)
    boot = {k: np.empty(n_boot) for k in keys}
    rng = np.random.default_rng(boot_seed)
    chunk = 100
    for c0 in range(0, n_boot, chunk):
        m = min(chunk, n_boot - c0)
        W = rng.multinomial(S, np.full(S, 1.0 / S), size=m).astype(np.float32)  # resampled session counts
        for k in keys:
            cm = (W @ sess_cm[k]).reshape(m, 4, 4)
            boot[k][c0:c0 + m] = f1_from_cm(cm)[0]
    log(f"Bootstrap done: {n_boot} resamples of {S:,} test sessions")

    def ci(a):
        return [float(np.percentile(a, 2.5)), float(np.percentile(a, 97.5))]

    def deltas_vs(refname):
        table = {}
        if refname not in results:
            return table
        ref = results[refname]
        ref_rep = boot[(refname, ref["representative_seed"])]
        ref_mean = np.mean([boot[(refname, s)] for s in ref["seeds"]], axis=0)
        for model, r in results.items():
            if model == refname:
                continue
            d_rep = boot[(model, r["representative_seed"])] - ref_rep
            d_avg = np.mean([boot[(model, s)] for s in r["seeds"]], axis=0) - ref_mean
            pt_rep = r["per_seed"][r["representative_seed"]]["macro_f1"] - ref["per_seed"][ref["representative_seed"]]["macro_f1"]
            pt_avg = r["macro_f1"]["mean"] - ref["macro_f1"]["mean"]
            table[model] = {
                "delta_rep_seed": pt_rep, "ci95_rep_seed": ci(d_rep),
                "p_delta_ge_0_rep": float((d_rep >= 0).mean()),
                "delta_seed_avg": pt_avg, "ci95_seed_avg": ci(d_avg),
                "p_delta_ge_0_avg": float((d_avg >= 0).mean()),
            }
        return table

    boot_table = deltas_vs(reference)
    boot_all = {r: deltas_vs(r) for r in [reference] + list(extra_references) if r in results}
    for model, r in results.items():
        r["macro_f1_ci95_rep_seed"] = ci(boot[(model, r["representative_seed"])])

    dataset = D.meta.get("dataset", "GA4 obfuscated e-commerce sample")
    out = {"title": title or f"Controlled comparison ({dataset}): test results", "dataset": dataset,
           "reference_model": reference, "n_boot": n_boot, "n_test_sessions": int(S),
           "n_test_prefixes": int(len(test)), "n_undetermined_test_prefixes": int(und.sum()),
           "models": results, "bootstrap_vs_reference": boot_table, "bootstrap_vs": boot_all,
           "leakage": D.meta.get("leakage"), "repro_all_pass": D.meta.get("repro_all_pass")}
    with open(os.path.join(out_dir, "results.json"), "w") as f:
        json.dump(out, f, indent=2, default=float)

    # ---- combined predictions parquet --------------------------------------------------
    comb = pd.DataFrame({"session_id": D.session_id[test], "prefix_len": plen, "y_true": y,
                         "undetermined": und})
    for (model, seed), p in preds_by.items():
        comb[f"{model}__seed{seed}"] = p.astype(np.int8)
    comb.to_parquet(os.path.join(out_dir, "test_predictions_all_models.parquet"), index=False)

    md = summary_markdown(out)
    with open(os.path.join(out_dir, "summary.md"), "w", encoding="utf-8") as f:
        f.write(md)
    return out, md


def _fmt(a):
    return f"{a['mean']:.4f} ± {a['std']:.4f}"


def summary_markdown(out):
    res = out["models"]
    order = sorted(res, key=lambda m: -res[m]["macro_f1"]["mean"])
    L = [f"# {out.get('title', 'Controlled comparison: test results')}", "",
         f"Dataset: {out.get('dataset', 'GA4 obfuscated e-commerce sample')}.", "",
         f"Test set: {out['n_test_prefixes']:,} prefixes from {out['n_test_sessions']:,} sessions. "
         f"Mean ± std over seeds. Bootstrap: {out['n_boot']} resamples of test sessions, "
         f"deltas vs `{out['reference_model']}`.", "",
         "## Macro-F1 and per-class F1", "",
         "| Model | Seeds | Macro-F1 | Browser | Researcher | Intent | Buyer | Undetermined | Runtime (min) |",
         "|---|---|---|---|---|---|---|---|---|"]
    for m in order:
        r = res[m]
        pc = r["per_class_f1"]
        L.append(f"| {m} | {r['n_seeds']} | {_fmt(r['macro_f1'])} | {pc['Browser']['mean']:.4f} | "
                 f"{pc['Researcher']['mean']:.4f} | {pc['Intent']['mean']:.4f} | {pc['Buyer']['mean']:.4f} | "
                 f"{_fmt(r['undetermined_macro_f1'])} | {r['runtime_sec_total'] / 60:.1f} |")
    tables = out.get("bootstrap_vs") or {out["reference_model"]: out["bootstrap_vs_reference"]}
    for refname, tab in tables.items():
        if not tab:
            continue
        L += ["", f"## Paired bootstrap: delta Macro-F1 vs {refname}", "",
              "| Model | Δ (rep. seed) | 95% CI | Δ (seed-avg) | 95% CI | P(Δ ≥ 0), seed-avg |",
              "|---|---|---|---|---|---|"]
        for m in order:
            b = tab.get(m)
            if b is None:
                continue
            L.append(f"| {m} | {b['delta_rep_seed']:+.4f} | [{b['ci95_rep_seed'][0]:+.4f}, {b['ci95_rep_seed'][1]:+.4f}] | "
                     f"{b['delta_seed_avg']:+.4f} | [{b['ci95_seed_avg'][0]:+.4f}, {b['ci95_seed_avg'][1]:+.4f}] | "
                     f"{b['p_delta_ge_0_avg']:.3f} |")
    L += ["", "## Macro-F1 by prefix length (mean over seeds)", "",
          "| Model | " + " | ".join(BIN_LABELS) + " |", "|---|" + "---|" * len(BIN_LABELS)]
    for m in order:
        L.append(f"| {m} | " + " | ".join(f"{res[m]['bins'][b]['mean']:.3f}" for b in BIN_LABELS) + " |")
    lk = out.get("leakage")
    if lk:
        L += ["", "## Leakage check (Exp3 item join on timestamp + bundle id)", "",
              f"- Events sharing a timestamp with another event of the same session: "
              f"{lk['events_sharing_timestamp_within_session']:.1%}"]
        for k in ("unique_items_sees_future_items", "unique_items_value_differs",
                  "graph_has_future_item_edges", "graph_has_future_add_to_cart_or_purchase_items"):
            v = lk[k]
            L.append(f"- `{k}`: {v['all']:,} of {lk['prefixes_total']:,} prefixes "
                     f"(test: {v['by_split']['test']:,})")
        L.append(f"- Exp3 graph edges not from the item's own event: {lk['exp3_edges_not_from_own_event']:,}; "
                 f"with the wrong relation type: {lk['exp3_edges_with_wrong_relation_type']:,}")
    return "\n".join(L) + "\n"


In [ ]:
sys.path.insert(0, "/content")
import importlib, exp4lib
from exp4lib import data, coveo, batching, models, train, baselines, evaluate
importlib.reload(exp4lib)
for _m in (data, coveo, batching, models, train, baselines, evaluate):
    importlib.reload(_m)
print("exp4lib loaded, version", exp4lib.__version__)
assert exp4lib.__version__ == NOTEBOOK_VERSION, "Old library code in /content/exp4lib: re-run the %%writefile cells above."


## Stage 1: Data

Unzips `browsing_train.csv` and `sku_to_content.csv` to local disk. DuckDB then streams the 36M-row file and selects
the most recent `N_SESSIONS` sessions of a contiguous period. The notebook builds the Exp4 tables (events, prefixes,
sessions, items, edges) and prints the class and split counts. Expect about 10–20 min (unzip + one pass over the CSV).

In [ ]:
if not os.path.exists(os.path.join(PREP_DIR, "meta.json")):
    t0 = time.time()
    meta = coveo.build_dataset(RAW_DIR, WORK_DIR, PREP_DIR, n_sessions=N_SESSIONS, end_gap_days=END_GAP_DAYS)
    log(f"Stage 1 done in {(time.time() - t0) / 60:.1f} min")
else:
    log("Stage 1 outputs found on Drive, skipping.")
with open(os.path.join(PREP_DIR, "meta.json")) as f:
    meta = json.load(f)
print("T_MAX:", meta["T_MAX"], "| sample:", meta["sample"])
print(json.dumps(meta["counts"], indent=2))

## Stage 2: Markov-3 and LightGBM

In [ ]:
import pandas as pd, gc
D = data.Exp4Data(PREP_DIR)
sessions_df = pd.read_parquet(os.path.join(PREP_DIR, "sessions.parquet"))
print({k: len(v) for k, v in D.rows.items()}, "class weights:", D.class_weights())

baselines.run_markov(D, sessions_df, RUN_DIR, n=3, log=log)
events_df = pd.read_parquet(os.path.join(PREP_DIR, "events.parquet"))
feats = baselines.build_lgbm_features(D, events_df, sessions_df,
                                      os.path.join(PREP_DIR, "lgbm_features.parquet"), log=log)
baselines.run_lightgbm(D, feats, sessions_df, RUN_DIR, seeds=SEEDS, n_trials=LGBM_TRIALS,
                       tune_frac=LGBM_TUNE_FRAC, timeout_sec=LGBM_TUNE_TIMEOUT_SEC, log=log)
del feats, events_df; gc.collect()
for p in sorted(os.listdir(os.path.join(RUN_DIR, "runs"))):
    if p.startswith(("Markov", "LightGBM__")):
        r = json.load(open(os.path.join(RUN_DIR, "runs", p)))
        print(f"{p[:-5]:28s} val={r['best_val_macro_f1']:.4f} test={r['test_macro_f1']:.4f}")

## Stage 3: Neural models (focused subset)

Seed by seed, epoch checkpoints on Drive, resumable. Prints the estimated GPU time left after each run.

In [ ]:
if "D" not in globals():
    D = data.Exp4Data(PREP_DIR)
jobs = [(s, m) for s in SEEDS for m in NEURAL_MODELS]
t_stage = time.time()
for seed, name in jobs:
    if train.is_done(RUN_DIR, name, seed) or train.reuse_from_lr_final(RUN_DIR, LR_ROOT, name, seed, log=log):
        continue
    train.train_one(D, name, seed, RUN_DIR, DEVICE, log=log)
    torch.cuda.empty_cache()
    times = {}
    for s, m in jobs:
        p = os.path.join(RUN_DIR, "runs", f"{m}__seed{s}.json")
        if os.path.exists(p):
            times.setdefault(m, []).append(json.load(open(p))["runtime_sec"])
    left = sum(sum(times[m]) / len(times[m]) if m in times else float("nan")
               for s, m in jobs if not train.is_done(RUN_DIR, m, s))
    log(f"Estimated GPU time left: {left / 3600:.1f} h (nan = some models not timed yet)")
log(f"Stage 3 finished ({(time.time() - t_stage) / 3600:.1f} h in this session)")

## Stage 4: Evaluation

Same metrics as Exp4: Macro-F1 mean ± std and per-class F1; paired bootstrap over test sessions (1,000 resamples)
vs `HGNN_full`, vs `GRU+Time+SF+PageItem` and vs `HomoGraphSAGE_3layer_meanpool`; prefix-length bins; undetermined
prefixes (no add/purchase and fewer than 3 detail events). Outputs: `results/results.json`, `results/summary.md`.

In [ ]:
if "D" not in globals():
    D = data.Exp4Data(PREP_DIR)
out, md_text = evaluate.evaluate_all(D, RUN_DIR, RESULTS_DIR, n_boot=N_BOOT, log=log)
from IPython.display import Markdown, display
display(Markdown(md_text))

## Stage 5 (optional): learning-rate selection

Robustness check of the training protocol. For each neural model, the learning rate is chosen from
`LR_GRID` on **validation** Macro-F1 with seed 42, then all seeds are trained at the chosen rate. Patience,
batch size, epochs, class weights and data are unchanged. Existing runs are reused: the default rate (1e-3)
comes from stage 3, so only the lower rates are trained. Results are written separately
(`lr_selection/final/`, `results_lr/`); the stage 4 results stay untouched. Resumable like every other stage.

In [ ]:
if "D" not in globals():
    D = data.Exp4Data(PREP_DIR)
RESULTS_LR_DIR = os.path.join(DRIVE_DIR, "results_lr")
t_stage = time.time()
choice = train.lr_selection(D, LR_MODELS, RUN_DIR, LR_ROOT, DEVICE, lrs=LR_GRID,
                            select_seed=SEEDS[0], seeds=LR_SEEDS, log=log)
log(f"Stage 5 training finished ({(time.time() - t_stage) / 3600:.1f} h in this session)")
print(json.dumps(choice, indent=2))
out_lr, md_lr = evaluate.evaluate_all(D, os.path.join(LR_ROOT, "final"), RESULTS_LR_DIR, n_boot=N_BOOT,
                                      title=f"Controlled comparison ({D.meta.get('dataset', 'GA4')}), "
                                            "learning rate selected on validation: test results", log=log)
with open(os.path.join(RESULTS_LR_DIR, "lr_choice.json"), "w") as f:
    json.dump(choice, f, indent=2)
from IPython.display import Markdown, display
display(Markdown(md_lr))